# ViBioMIR – Data Pipeline (Phiên bản kỹ sư v3.1)
> **R2AI 2026 – ViBioMIR** | Đặc tả tham chiếu: `PHAN_CONG.md`, `HUONG_DAN_CRAWL.md`, `DAC_TA_LAM_SACH_v1.md`

| Giai đoạn | Mã | Nội dung | Đầu ra chính |
|---|:---:|---|---|
| **Bước 1** | **T0.1** | Tải URL từ HF & EDA | `raw/*.parquet`, `reports/corpus_stats.md` |
| **Bước 2** | **T0.2 / T1.1** | Async crawler + DomainLimiter + PubMed Europe PMC | `crawl/status.duckdb`, `crawl/html/shard_*.jsonl.zst`, `crawl/pubmed/*.json.zst` |
| **Bước 3** | **T1.2** | Clean & trích xuất văn bản theo luật domain | Blocks + text nguyên văn |
| **Bước 4** | **T1.3** | Dedup (sha1 + MinHash) & xuất Parquet schema v2 | `docs/run_<run_id>/docs_*.parquet`, `reports/quality_dashboard.md` |

**Nguyên tắc cốt tử**

> Code đã được cập nhật theo các đặc tả; đây không phải xác nhận extractor đã qua cổng Gold. Chưa chạy crawl/API trong lượt sửa này.
1. **Nguyên văn 100%** – không viết lại, không sửa chính tả, không NFC/NFKC.
2. **Không mất ID** – dedup chỉ gán `dup_group`, không xóa bản ghi.
3. **Crawl có trách nhiệm** – rate limit per-domain, retry backoff, tôn trọng server.

**Thay đổi v3.1**
- Kiến trúc worker-pool + token-bucket per-domain, không round-robin thủ công.
- Graceful shutdown qua SIGINT/SIGTERM hoặc file `STOP`; URL chưa xử lý giữ pending.
- Resume không cần checkpoint queue: trạng thái nằm trong DuckDB.
- Shard ghi atomic; DB ghi theo batch transaction.
- Structured logging, per-domain metrics, health check cuối run.
**Lượt PubMed ưu tiên (21/10):** Cell API được đặt trước crawler HTML. Sau phần setup/tải corpus ở đầu notebook, chạy cell `PubMed qua Europe PMC REST API`. Cell này dùng lại mọi cache PMID đã có trong `crawl/pubmed/` trên Drive và chỉ gửi `searchPOST` cho PMID còn thiếu (tối đa 1.000 PMID/request). Không cần chạy cell crawler trước. Crawl HTML mặc định `RUN_CRAWL=True`: chạy lại cell `Orchestrator: run_crawler_pipeline`, rồi chạy cell chọn URL và cell chạy crawler. Sau mỗi lượt crawl, chạy cell `Trạng thái toàn corpus theo domain` và xem bảng trước khi kết luận.


## 0. Cài đặt thư viện


In [ ]:
import subprocess, sys

COLAB = 'google.colab' in sys.modules

required_packages = [
    'huggingface_hub',
    'pandas>=2.0', 'pyarrow',
    'duckdb>=0.9.0',
    'httpx[http2]>=0.25.0',
    'aiofiles',
    'charset-normalizer',
    'selectolax',
    'trafilatura',
    'python-dotenv',
    'tqdm', 'lxml',
    'nest_asyncio',
    'zstandard',
    'tabulate',
]

if COLAB:
    print('Cài đặt thư viện cho Google Colab...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + required_packages, check=True)
else:
    missing = []
    for pkg in required_packages:
        name = pkg.split('>=')[0].split('[')[0]
        name = {'python-dotenv': 'dotenv', 'charset-normalizer': 'charset_normalizer'}.get(name, name)
        try:
            __import__(name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f'Cài thư viện thiếu: {missing}')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=True)
    print('Môi trường Local đã đủ thư viện.')


## 1. Imports, cấu hình đường dẫn, logging


In [ ]:
import os, sys, re, json, hashlib, asyncio, base64, time, signal, warnings, shutil, random
from pathlib import Path
from urllib.parse import urlparse
from datetime import datetime, timezone
from typing import Optional, Dict, Any, List, Tuple
from collections import Counter, defaultdict
import logging

import pandas as pd
import duckdb
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning)

# ---- CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU NGOÀI REPO ----
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/r2ai_data')
else:
    DATA_ROOT = Path(os.environ.get('R2AI_DATA_ROOT', 'D:/r2ai_data')).expanduser().resolve()

RAW_DIR     = DATA_ROOT / 'raw'
CRAWL_DIR   = DATA_ROOT / 'crawl'
HTML_DIR    = CRAWL_DIR / 'html'
PUBMED_DIR  = CRAWL_DIR / 'pubmed'
DOCS_DIR    = DATA_ROOT / 'docs'
REPORT_DIR  = DATA_ROOT / 'reports'
AUDIT_DIR   = REPORT_DIR / 'audit'
LOG_DIR     = DATA_ROOT / 'logs'

for d in [RAW_DIR, CRAWL_DIR, HTML_DIR, PUBMED_DIR, DOCS_DIR, REPORT_DIR, AUDIT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'DATA_ROOT = {DATA_ROOT}')
for label, p in [('raw', RAW_DIR), ('crawl', CRAWL_DIR), ('docs', DOCS_DIR), ('reports', REPORT_DIR), ('logs', LOG_DIR)]:
    print(f'  {label:8s} -> {p}')


In [ ]:
# ---- LOGGING: console + file xoay vòng theo ngày ----
log_file = LOG_DIR / f'pipeline_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'

logger = logging.getLogger('vibiomir')
logger.setLevel(logging.INFO)
logger.handlers.clear()

fmt = logging.Formatter('%(asctime)s | %(levelname)-7s | %(name)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S')

ch = logging.StreamHandler(sys.stdout)
ch.setFormatter(fmt)
logger.addHandler(ch)

fh = logging.FileHandler(log_file, encoding='utf-8')
fh.setFormatter(fmt)
logger.addHandler(fh)

logger.info(f'Log file: {log_file}')


In [ ]:
# ---- .env ----
from dotenv import load_dotenv

for p in [Path('.env'), Path('../.env'), DATA_ROOT / '.env']:
    if p.exists():
        load_dotenv(p)
        logger.info(f'Đọc .env từ: {p}')
        break
else:
    logger.info('Không có .env – OK với dataset public.')

HF_TOKEN = os.getenv('HF_TOKEN')
logger.info('HF_TOKEN: ' + ('đã nạp' if HF_TOKEN else 'không có (dataset public)'))


---
## Bước 1 — Tải URL từ Hugging Face & EDA (T0.1)


In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO = 'AIGuruTinix/ViBioMIR'
FILES   = ['links_corpus_vi_en_zh.parquet', 'query.parquet']

logger.info(f'Tải dataset từ {HF_REPO}')
for fname in FILES:
    dest = RAW_DIR / fname
    if dest.exists() and dest.stat().st_size > 0:
        logger.info(f'[đã có] {fname} ({dest.stat().st_size/1e6:.2f} MB)')
        continue
    logger.info(f'Đang tải {fname}...')
    hf_hub_download(repo_id=HF_REPO, filename=fname, repo_type='dataset', token=HF_TOKEN, local_dir=str(RAW_DIR))
    logger.info(f'Xong {fname} ({dest.stat().st_size/1e6:.2f} MB)')

df_links   = pd.read_parquet(RAW_DIR / 'links_corpus_vi_en_zh.parquet')
df_queries = pd.read_parquet(RAW_DIR / 'query.parquet')

print(f'links_corpus : {len(df_links):,} | cols={list(df_links.columns)}')
print(f'queries      : {len(df_queries):,} | cols={list(df_queries.columns)}')
df_links.head(3)


In [ ]:
# ---- Kiểm tra toàn vẹn T0.1 ----
assert df_links['id'].dtype == 'int64'
assert df_links['id'].duplicated().sum() == 0
assert df_links['id'].isna().sum() == 0
assert df_links['url'].isna().sum() == 0
logger.info('Kiểm tra ID: OK')
print(f'Tổng URL: {len(df_links):,} | ID range: {df_links["id"].min():,} → {df_links["id"].max():,}')


In [ ]:
# ---- EDA domain ----
df_links['domain'] = df_links['url'].apply(lambda u: urlparse(u).netloc.lower().removeprefix('www.'))

domain_stats = (
    df_links.groupby('domain').agg(n_urls=('id','count'))
    .sort_values('n_urls', ascending=False).reset_index()
)
domain_stats['pct']     = (domain_stats['n_urls'] / len(df_links) * 100).round(3)
domain_stats['cum_pct'] = domain_stats['pct'].cumsum().round(3)

df_links['ext'] = df_links['url'].str.extract(r'\.([a-zA-Z0-9]{2,5})(?:[?#]|$)', expand=False).str.lower()
non_html_exts = ['pdf','doc','docx','xls','xlsx','zip','mp4','jpg','png','gif','svg']
mask_non_html = df_links['ext'].isin(non_html_exts)

print(f'Số domain: {len(domain_stats):,}')
print(f'Non-HTML: {mask_non_html.sum():,} ({mask_non_html.mean()*100:.2f}%)')
domain_stats.head(20)


In [ ]:
# ---- Xuất báo cáo T0.1 ----
report_lines = [
    '# Corpus Stats – T0.1',
    f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
    '',
    '## 1. Tổng quan',
    f'- URLs: {len(df_links):,}',
    f'- Queries: {len(df_queries):,}',
    f'- Domains: {len(domain_stats):,}',
    f'- ID min/max: {df_links["id"].min():,} / {df_links["id"].max():,}',
    f'- Non-HTML: {mask_non_html.sum():,} ({mask_non_html.mean()*100:.2f}%)',
    '',
    '## 2. Top 50 domain',
    domain_stats.head(50).to_markdown(index=False),
]
(REPORT_DIR / 'corpus_stats.md').write_text('\n'.join(report_lines), encoding='utf-8')
logger.info(f'Đã ghi báo cáo: {REPORT_DIR / "corpus_stats.md"}')


## Ưu tiên 1 — hoàn tất PubMed trước crawl HTML

Chạy sau setup và tải corpus. Tái sử dụng mọi batch cache trong `crawl/pubmed/`; chỉ gọi Europe PMC `searchPOST` cho PMID thiếu, tối đa 1.000 PMID/request. Không cần mở DuckDB hoặc chạy crawler. Xem `reports/pubmed_coverage.csv` và `pubmed_missing_pmids.csv` sau khi chạy.

Dừng bằng `crawl/STOP`; xóa file STOP có chủ ý trước khi tiếp tục. Batch đã hoàn tất được giữ để resume.


In [ ]:
# ---- PubMed qua Europe PMC REST API ----
import httpx
import zstandard as zstd

pubmed_links = df_links[df_links['url'].str.contains('pubmed.ncbi.nlm.nih.gov', na=False)].copy()
pubmed_links['pmid'] = pubmed_links['url'].str.extract(r'/(\d+)/?$')
pubmed_links = pubmed_links.dropna(subset=['pmid'])
pubmed_links['pmid'] = pubmed_links['pmid'].astype(str)
print(f'PubMed cần lấy: {len(pubmed_links):,}')

EUROPE_PMC_POST_URL = 'https://www.ebi.ac.uk/europepmc/webservices/rest/searchPOST'
RUN_PUBMED = True               # review 20/10: lấy toàn bộ PMID còn thiếu; cache batch được tái sử dụng
PUBMED_MAX_BATCHES: Optional[int] = None  # None = toàn bộ; số nguyên = chạy pilot
pubmed_records: List[Dict[str, Any]] = []


def fetch_pubmed_batches(pmid_list: List[str], batch_size: int = 1000,
                         max_batches: Optional[int] = None) -> List[Dict[str, Any]]:
    """Dùng lại cache cũ/mới theo PMID; chỉ searchPOST các PMID chưa có."""
    if not 1 <= batch_size <= 1000:
        raise ValueError('batch_size must be between 1 and 1000 PMID')
    if max_batches is not None and max_batches <= 0:
        raise ValueError('max_batches must be positive or None')
    pmids = list(dict.fromkeys(str(p).strip() for p in pmid_list if str(p).strip()))
    if max_batches is not None:
        pmids = pmids[:max_batches * batch_size]
    wanted = set(pmids)
    PUBMED_DIR.mkdir(parents=True, exist_ok=True)
    cctx = zstd.ZstdCompressor(level=3, threads=-1)
    dctx = zstd.ZstdDecompressor()
    by_pmid: Dict[str, Dict[str, Any]] = {}
    for cache_path in sorted(PUBMED_DIR.glob('batch_*.json.zst')):
        try:
            cached = json.loads(dctx.decompress(cache_path.read_bytes()).decode('utf-8'))
            if not isinstance(cached, list):
                raise ValueError('cache không phải danh sách')
            for hit in cached:
                if isinstance(hit, dict):
                    pmid = str(hit.get('pmid', '')).strip()
                    if pmid in wanted:
                        by_pmid[pmid] = hit
        except Exception as exc:
            logger.warning(f'Bỏ qua cache PubMed lỗi {cache_path.name}: {exc}')
    pending = [p for p in pmids if p not in by_pmid]
    logger.info(f'PubMed cache: {len(by_pmid):,}/{len(pmids):,} PMID; cần searchPOST {len(pending):,}; '
                f'tối thiểu {(len(pending) + batch_size - 1) // batch_size:,} request chưa tính retry/phân trang')
    if not pending:
        return list(by_pmid.values())
    delays = [1.0, 3.0, 10.0, 30.0]

    with httpx.Client(timeout=60.0, http2=True) as client:
        for b_idx, start in enumerate(tqdm(range(0, len(pending), batch_size), desc='Europe PMC')):
            if (CRAWL_DIR/'STOP').exists():
                raise KeyboardInterrupt('STOP requested; PubMed cache batches already saved.')
            chunk = pending[start:start + batch_size]
            digest = hashlib.sha1('|'.join(chunk).encode('utf-8')).hexdigest()[:10]
            cache_path = PUBMED_DIR / f'batch_{b_idx:05d}_{digest}.json.zst'
            query_str = f'EXT_ID:({" OR ".join(chunk)}) AND SRC:MED'
            cursor_mark = '*'
            batch_hits: Dict[str, Dict[str, Any]] = {}
            while True:
                data = {'query': query_str, 'resultType': 'core', 'format': 'json',
                        'pageSize': str(batch_size), 'cursorMark': cursor_mark}
                payload = None
                last_error = None
                for req_attempt, delay in enumerate(delays):
                    if (CRAWL_DIR/'STOP').exists():
                        raise KeyboardInterrupt('STOP requested; PubMed cache batches already saved.')
                    try:
                        resp = client.post(EUROPE_PMC_POST_URL, data=data)
                        resp.raise_for_status()
                        payload = resp.json()
                        break
                    except Exception as exc:
                        last_error = exc
                        if req_attempt < len(delays) - 1:
                            logger.warning(f'PubMed batch {b_idx} request {req_attempt + 1} lỗi; '
                                           f'retry sau {delay:g}s: {exc}')
                            time.sleep(delay)
                if payload is None:
                    raise RuntimeError(f'PubMed batch {b_idx} thất bại sau {len(delays)} lần') from last_error
                page_hits = payload.get('resultList', {}).get('result', []) or []
                for hit in page_hits:
                    pmid = str(hit.get('pmid', '')).strip()
                    if pmid in chunk:
                        batch_hits[pmid] = hit
                next_cursor = payload.get('nextCursorMark')
                time.sleep(1.0)
                if (len(batch_hits) == len(chunk) or not page_hits
                        or not next_cursor or next_cursor == cursor_mark):
                    break
                cursor_mark = next_cursor

            batch_list = list(batch_hits.values())
            tmp_path = cache_path.with_suffix(cache_path.suffix + '.tmp')
            tmp_path.write_bytes(cctx.compress(json.dumps(batch_list, ensure_ascii=False).encode('utf-8')))
            os.replace(tmp_path, cache_path)
            by_pmid.update(batch_hits)
            logger.info(f'PubMed batch {b_idx}: {len(batch_list):,}/{len(chunk):,} PMID; '
                        f'tổng có {len(by_pmid):,}/{len(pmids):,}')
    return list(by_pmid.values())


if RUN_PUBMED:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    pubmed_records = fetch_pubmed_batches(
        pubmed_links['pmid'].tolist(), batch_size=1000, max_batches=PUBMED_MAX_BATCHES
    )
    requested_list = list(dict.fromkeys(pubmed_links['pmid'].astype(str)))
    if PUBMED_MAX_BATCHES is not None:
        requested_list = requested_list[:PUBMED_MAX_BATCHES * 1000]
    requested_pmids = set(requested_list)
    returned_pmids = {str(r.get('pmid', '')).strip() for r in pubmed_records if r.get('pmid')}
    missing_pmids = requested_pmids - returned_pmids
    logger.info(f'PubMed: requested={len(requested_pmids):,} | returned={len(returned_pmids):,} | missing={len(missing_pmids):,}')
    if missing_pmids:
        logger.warning(f'Retry {len(missing_pmids):,} PMID chưa có trong kết quả')
        retry_records = fetch_pubmed_batches(sorted(missing_pmids), batch_size=1000, max_batches=None)
        by_pmid = {str(r.get('pmid')): r for r in pubmed_records if r.get('pmid')}
        by_pmid.update({str(r.get('pmid')): r for r in retry_records if r.get('pmid')})
        pubmed_records = list(by_pmid.values())
        still_missing = requested_pmids - {str(r.get('pmid')) for r in pubmed_records if r.get('pmid')}
        logger.warning(f'PubMed còn thiếu sau retry: {len(still_missing):,}')
    missing_final = requested_pmids - {str(r.get('pmid')) for r in pubmed_records if r.get('pmid')}
    pd.DataFrame({'pmid':sorted(missing_final)}).to_csv(REPORT_DIR/'pubmed_missing_pmids.csv',index=False)
    pd.DataFrame([dict(requested=len(requested_pmids),returned=len(requested_pmids)-len(missing_final),missing=len(missing_final))]).to_csv(REPORT_DIR/'pubmed_coverage.csv',index=False)
else:
    print('RUN_PUBMED=False; chưa gọi Europe PMC. Bật True để lấy dữ liệu theo cấu hình.')


---
## Bước 2 — Crawler production-grade (T0.2 / T1.1)

### Kiến trúc
1. **DuckDB status** – bảng `crawl_status` là nguồn trạng thái; resume lấy pending và lỗi tạm thời còn ngân sách attempts.
2. **DomainLimiter** – token bucket per-domain + semaphore giới hạn concurrency.
3. **Worker pool** – N worker cố định, đọc từ `asyncio.Queue`; domain limiter lo phân phối tải.
4. **Retry** – backoff `5s → 30s → 120s` (jitter ±20%); lỗi 404/410 dừng, 403/521 dừng (blocked).
5. **Adaptive backoff** – 429/503 làm domain đó pause 5 phút; sau đó tốc độ giảm tối đa một nửa so với ban đầu, không nhân đôi interval vô hạn.
6. **Shard writer** – ghi file tạm `.tmp` rồi `os.replace()`; tối đa 10.000 response/shard, gồm HTML và non-HTML HTTP 200.
7. **Status flusher** – lỗi không có raw được upsert theo batch 1.000; status có raw chỉ commit sau khi shard đã ghi xong.
8. **Graceful shutdown** – SIGINT/SIGTERM hoặc file `STOP` → drain hàng đã enqueue, flush shard/status; phần còn lại giữ pending.
9. **Skip domain đặc biệt** – `pubmed.ncbi.nlm.nih.gov` (API riêng), `zysjonline.com` (đã có), `120ask.com` (cloud riêng).
10. **Domain gate** – 1 request/domain trước khi xếp hàng; domain trả 403/challenge giữ toàn bộ URL ở `pending` và ghi vào `domain_gate`, không đốt `attempts` cũng không ghi `blocked` giả.


In [ ]:
# ---- DuckDB setup ----
# Local: dùng DB đã có ở gốc repo khi chưa đồng bộ vào DATA_ROOT/crawl.
_local_db = Path.cwd() / 'status.duckdb'
_default_db = CRAWL_DIR / 'status.duckdb'
if not COLAB and not _default_db.is_file() and _local_db.is_file():
    _default_db = _local_db
DB_PATH = Path(os.environ.get('R2AI_STATUS_DB', str(_default_db))).expanduser().resolve()
print(f'DB_PATH = {DB_PATH}')
con = duckdb.connect(str(DB_PATH))

con.execute("""
CREATE TABLE IF NOT EXISTS crawl_status (
    id            BIGINT PRIMARY KEY,
    url           VARCHAR,
    final_url     VARCHAR,
    lang_expected VARCHAR,
    status        VARCHAR DEFAULT 'pending',
    http_code     INTEGER,
    content_type  VARCHAR,
    n_bytes       BIGINT,
    attempts      INTEGER DEFAULT 0,
    fetched_at    TIMESTAMP,
    shard         INTEGER,
    sha1_raw      VARCHAR,
    error_msg     VARCHAR
)
""")

# Migrate schema nếu DB cũ chưa có các cột dùng bởi crawler mới.
existing_cols = {row[1] for row in con.execute("PRAGMA table_info('crawl_status')").fetchall()}
required_cols = {
    'final_url': 'VARCHAR', 'lang_expected': 'VARCHAR', 'status': 'VARCHAR',
    'http_code': 'INTEGER', 'content_type': 'VARCHAR', 'n_bytes': 'BIGINT',
    'attempts': 'INTEGER', 'fetched_at': 'TIMESTAMP', 'shard': 'INTEGER',
    'sha1_raw': 'VARCHAR', 'error_msg': 'VARCHAR',
}
for col, dtype in required_cols.items():
    if col not in existing_cols:
        con.execute(f'ALTER TABLE crawl_status ADD COLUMN {col} {dtype}')
        logger.info(f'Đã thêm cột thiếu: {col} ({dtype})')

# Index cho truy vấn resume nhanh
con.execute("CREATE INDEX IF NOT EXISTS idx_status ON crawl_status(status)")
logger.info(f'DuckDB ready: {DB_PATH}')
print('Số bản ghi hiện có:', con.execute('SELECT COUNT(*) FROM crawl_status').fetchone()[0])


In [ ]:
# ---- Đọc kế hoạch domain (tuỳ chọn) ----
DOMAIN_PLAN_PATH = Path('crawl/domain_plan.csv')
domain_plan_dict: Dict[str, Dict[str, Any]] = {}
if DOMAIN_PLAN_PATH.exists():
    df_plan = pd.read_csv(DOMAIN_PLAN_PATH)
    for _, r in df_plan.iterrows():
        dom = str(r['domain']).strip().lower().removeprefix('www.')
        domain_plan_dict[dom] = {
            'rate':   float(r.get('start_req_per_s', 2.0)),
            'access': str(r.get('access', 'ok')),
            'lang':   str(r.get('lang', 'vi')),
        }
    logger.info(f'Đọc {len(domain_plan_dict)} domain plan')

# Long Châu: nhịp thấp, nhận diện crawler trung thực; không giả lập trình duyệt/né WAF.
for _lc_dom in ('nhathuoclongchau.com.vn', 'tiemchunglongchau.com.vn'):
    domain_plan_dict.setdefault(_lc_dom, {'rate': 0.5, 'access': 'ok', 'lang': 'vi'})
    domain_plan_dict[_lc_dom]['rate'] = min(float(domain_plan_dict[_lc_dom].get('rate', 0.5)), 0.5)

def get_expected_lang(dom: str) -> str:
    dom = dom.lower().removeprefix('www.')
    if dom == '39.net' or dom.endswith('.39.net'): return 'zh'
    if 'pubmed' in dom: return 'en'
    if dom in domain_plan_dict: return domain_plan_dict[dom]['lang']
    if any(dom.endswith(s) for s in ('.cn', '.com.cn')): return 'zh'
    return 'vi'

In [ ]:
# ---- Seed toàn bộ URL vào DuckDB (rất nhanh, idempotent) ----
if 'domain' not in df_links.columns:
    df_links['domain'] = df_links['url'].apply(lambda u: urlparse(u).netloc.lower().removeprefix('www.'))
df_links['lang_expected'] = df_links['domain'].map(get_expected_lang)

before = con.execute('SELECT COUNT(*) FROM crawl_status').fetchone()[0]
con.execute("""
    INSERT OR IGNORE INTO crawl_status (id, url, lang_expected, status, attempts)
    SELECT id, url, lang_expected, 'pending', 0 FROM df_links
""")
after = con.execute('SELECT COUNT(*) FROM crawl_status').fetchone()[0]
logger.info(f'Seed xong: {after - before:+,} bản ghi mới | tổng {after:,}')
print(f'Tổng bản ghi trong DuckDB: {after:,}')

# INSERT OR IGNORE không sửa metadata của ID đã seed từ lượt cũ.
con.execute("""UPDATE crawl_status SET lang_expected='zh'
    WHERE regexp_replace(lower(split_part(url, '/', 3)), '^www[.]', '')='39.net'
       OR regexp_replace(lower(split_part(url, '/', 3)), '^www[.]', '') LIKE '%.39.net'
""")
con.execute('CHECKPOINT')


In [ ]:
# ---- Encoding helper ----
from charset_normalizer import from_bytes

_CHARSET_RE = re.compile(r'charset=[\"\']?([\w-]+)', re.I)

def detect_encoding(raw_bytes: bytes, content_type: str = '') -> str:
    """Content-Type → <meta charset> → charset_normalizer. gb2312/gbk → gb18030."""
    if content_type:
        m = _CHARSET_RE.search(content_type)
        if m:
            enc = m.group(1).lower()
            return 'gb18030' if enc in ('gb2312','gbk','x-gbk') else enc
    head = raw_bytes[:4096].decode('ascii', errors='replace')
    m = _CHARSET_RE.search(head)
    if m:
        enc = m.group(1).lower()
        return 'gb18030' if enc in ('gb2312','gbk','x-gbk') else enc
    res = from_bytes(raw_bytes[:32768]).best()
    return res.encoding if res else 'utf-8'

def extract_laodong_cookie(body_text: str) -> Optional[Dict[str, str]]:
    m = re.search(r'document\.cookie\s*=\s*[\"\']D1N=([^\'\"\;]+)', body_text)
    return {'D1N': m.group(1).strip()} if m else None


In [ ]:
# ---- DomainLimiter: token bucket + pause khi 429/503 ----
class DomainLimiter:
    """
    - rate (req/s) per-domain, tính lại interval khi slow_down.
    - concurrency: số request đồng thời cho domain này.
    - pause_until: tạm dừng toàn bộ domain (khi nhận 429/503).
    - stats: đếm ok/fail/bytes/latency để làm báo cáo.
    """
    def __init__(self, domain: str, rate: float = 2.0, concurrency: int = 4):
        self.domain = domain
        self.rate = float(rate)
        if self.rate <= 0 or concurrency <= 0:
            raise ValueError('rate and concurrency must be positive')
        self.base_rate = self.rate
        self.base_interval = 1.0 / self.rate
        self.interval = self.base_interval
        self.sem = asyncio.Semaphore(concurrency)
        self.lock = asyncio.Lock()
        self.next_slot = 0.0
        self.pause_until = 0.0
        self.halted = False  # dừng domain đến lần chạy có chủ ý sau khi server chặn
        self.stats = {
            'attempts': 0, 'ok': 0, 'fail': 0, 'blocked': 0,
            'timeout': 0, 'bytes': 0, 'latencies': [],
            'last_error': None, 'last_error_at': None,
        }

    async def acquire(self):
        await self.sem.acquire()
        try:
            while True:
                async with self.lock:
                    if self.halted:
                        self.sem.release()
                        return False
                    now = time.monotonic()
                    wait = max(self.pause_until, self.next_slot) - now
                    if wait <= 0:
                        self.next_slot = now + self.interval
                        return True
                # Re-read halt/pause/interval after waking; do not use a stale reserved slot.
                await asyncio.sleep(min(wait, 1.0))
        except BaseException:
            self.sem.release()
            raise

    def release(self):
        self.sem.release()

    def slow_down(self, factor: float = 2.0, duration_sec: float = 300.0, halt: bool = False):
        """429/503: pause toàn domain và giảm một nửa tốc độ sau đó."""
        self.pause_until = max(self.pause_until, time.monotonic() + duration_sec)
        self.halted = self.halted or halt
        # Giảm còn tối đa 1/factor tốc độ ban đầu; các 429 đồng thời không nhân đôi vô hạn.
        self.interval = max(self.interval, self.base_interval * factor)
        logger.warning(f'[{self.domain}] slow_down: pause {duration_sec}s, interval -> {self.interval:.3f}s (base={self.base_interval:.3f}s)')

    def record(self, ok: bool, n_bytes: int, latency: float, status: str, error: Optional[str] = None):
        s = self.stats
        s['attempts'] += 1
        s['bytes'] += n_bytes or 0
        s['latencies'].append(latency)
        if ok: s['ok'] += 1
        else:
            s['fail'] += 1
            if status == 'blocked': s['blocked'] += 1
            if status == 'timeout': s['timeout'] += 1
            s['last_error'] = f'{status}: {error}' if error else status
            s['last_error_at'] = datetime.now(timezone.utc).isoformat()

    def snapshot(self) -> Dict[str, Any]:
        lats = self.stats['latencies']
        return {
            'domain': self.domain,
            'attempts': self.stats['attempts'],
            'ok': self.stats['ok'],
            'fail': self.stats['fail'],
            'blocked': self.stats['blocked'],
            'timeout': self.stats['timeout'],
            'bytes': self.stats['bytes'],
            'p50_ms': (sorted(lats)[len(lats)//2] * 1000) if lats else 0.0,
            'p95_ms': (sorted(lats)[int(len(lats)*0.95)] * 1000) if lats else 0.0,
            'last_error': self.stats['last_error'],
        }

class LimiterManager:
    def __init__(self, plan: Dict[str, Dict[str, Any]]):
        self.plan = plan
        self.limiters: Dict[str, DomainLimiter] = {}
        self.lock = asyncio.Lock()

    async def get_limiter(self, domain: str) -> DomainLimiter:
        is_39 = domain == '39.net' or domain.endswith('.39.net')
        if is_39:
            domain = '39.net'  # one budget shared by every subdomain
        async with self.lock:
            if domain not in self.limiters:
                cfg = self.plan.get(domain, {})
                rate = 1.0 if is_39 else cfg.get('rate', 2.0)
                # Một request đồng thời cho Long Châu; dừng domain khi server trả 429/403/challenge.
                conc = 1 if is_39 or 'longchau.com.vn' in domain else 4
                self.limiters[domain] = DomainLimiter(domain, rate=rate, concurrency=conc)
            return self.limiters[domain]

    def all_snapshots(self) -> List[Dict[str, Any]]:
        return [lim.snapshot() for lim in self.limiters.values()]

In [ ]:
# ---- Fetch với retry + jitter ----
import httpx, nest_asyncio
nest_asyncio.apply()

# Nhận diện crawler rõ ràng; không giả mạo Chrome hoặc gửi header né cơ chế truy cập.
REQUEST_HEADERS = {
    'User-Agent': 'R2AIResearchCrawler/1.0',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'vi-VN,vi;q=0.9,en;q=0.7',
}

RETRY_DELAYS = [5.0, 30.0, 120.0]
RETRYABLE_CODES = {429, 500, 502, 503, 504, 520, 522, 524}
MAX_TOTAL_ATTEMPTS = len(RETRY_DELAYS) + 1   # tối đa 3 lần retry sau lần đầu
PERMANENT_CODES = {400, 401, 403, 404, 405, 410, 451, 521}
TIMEOUT = httpx.Timeout(connect=15.0, read=30.0, write=10.0, pool=30.0)

def _jitter(x: float, pct: float = 0.2) -> float:
    return max(0.1, x * (1.0 + random.uniform(-pct, pct)))

def crawl_challenge_reason(url, final_url, raw_bytes, response_headers=None):
    final_lower = str(final_url).lower()
    if 'verify.html' in final_lower:
        return 'verification_redirect'
    origin = (urlparse(url).hostname or '').lower()
    final = (urlparse(final_url).hostname or '').lower()
    if origin and final and origin != final and len(raw_bytes) < 3000:
        return 'small_cross_host_redirect'
    if 'longchau.com.vn' in origin:
        headers = {str(k).lower(): str(v).lower() for k, v in (response_headers or {}).items()}
        if headers.get('cf-mitigated') == 'challenge':
            return 'anti_bot_challenge_header'
        sample = raw_bytes[:200_000].decode('utf-8', errors='ignore').lower()
        challenge_markers = (
            'cf-chl-', 'challenge-platform', 'just a moment',
            'checking your browser', 'verify you are human',
            'access denied', 'captcha', 'robot or human',
        )
        if any(marker in sample for marker in challenge_markers):
            return 'anti_bot_challenge_body'
    return None


async def fetch_with_retry(client: httpx.AsyncClient, manager: LimiterManager,
                           row: Dict[str, Any], shard_id: int) -> Tuple[Dict[str, Any], Optional[Dict[str, Any]]]:
    doc_id = int(row['id'])
    url    = row['url']
    dom    = row['domain']
    limiter = await manager.get_limiter(dom)
    prior_attempts = row.get('attempts', 0)
    prior_attempts = 0 if prior_attempts is None or pd.isna(prior_attempts) else max(0, int(prior_attempts))
    allowed_attempts = max(0, MAX_TOTAL_ATTEMPTS - prior_attempts)
    res = {
        'id': doc_id, 'url': url, 'final_url': url,
        'status': 'pending', 'http_code': None, 'content_type': None,
        'n_bytes': 0, 'attempts': prior_attempts,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'shard': shard_id, 'sha1_raw': None, 'error_msg': None,
    }
    raw_rec = None
    if allowed_attempts == 0:
        res['status'] = 'http_error'
        res['error_msg'] = f'Đã hết ngân sách {MAX_TOTAL_ATTEMPTS} request cho URL'
        return res, None

    latency = 0.0
    for attempt in range(allowed_attempts):
        acquired = await limiter.acquire()
        if not acquired:
            # Domain đã bị server từ chối (403/challenge) nên nó bị halt: URL này chưa hề gửi request.
            # Giữ 'pending' và không tăng attempts, thay vì ghi 'blocked' (giả) cho toàn bộ domain.
            res['status'] = 'pending'
            res['http_code'] = None
            res['error_msg'] = 'domain_halted: không gửi request, giữ pending'
            break
        res['attempts'] += 1
        t0 = time.monotonic()
        try:
            resp = await client.get(url, headers=REQUEST_HEADERS, follow_redirects=True, timeout=TIMEOUT)
            latency = time.monotonic() - t0
            res['http_code']    = resp.status_code
            res['final_url']    = str(resp.url)
            res['content_type'] = resp.headers.get('content-type', '')
            raw_bytes = resp.content
            res['n_bytes'] = len(raw_bytes)

            # laodong.vn challenge
            response_status = resp.status_code
            if dom == 'laodong.vn' and response_status == 200 and len(raw_bytes) < 500:
                cookie_val = extract_laodong_cookie(raw_bytes.decode('utf-8', errors='replace'))
                if cookie_val:
                    resp2 = await client.get(url, headers=REQUEST_HEADERS, cookies=cookie_val,
                                             follow_redirects=True, timeout=TIMEOUT)
                    raw_bytes = resp2.content
                    response_status = resp2.status_code
                    res['http_code']    = response_status
                    res['final_url']    = str(resp2.url)
                    res['content_type'] = resp2.headers.get('content-type', '')
                    res['n_bytes']      = len(raw_bytes)

            challenge = crawl_challenge_reason(url, res['final_url'], raw_bytes, resp.headers)
            if challenge:
                res['status'] = 'blocked'
                res['error_msg'] = challenge
                res['sha1_raw'] = hashlib.sha1(raw_bytes).hexdigest()
                limiter.slow_down(1.0, 300.0, halt=True)
                break

            if response_status == 200:
                ct = (res['content_type'] or '').lower()
                if 'text/html' not in ct and 'application/xhtml' not in ct:
                    res['status'] = 'non_html'
                elif len(raw_bytes) < 100:
                    res['status'] = 'empty'
                else:
                    res['status'] = 'ok'
                res['sha1_raw'] = hashlib.sha1(raw_bytes).hexdigest()
                enc = detect_encoding(raw_bytes, res['content_type'])
                # Long Chau: chỉ đưa HTML hợp lệ vào shard; status ok và raw phải đi cùng nhau.
                save_raw = (res['status'] == 'ok') if 'longchau.com.vn' in (urlparse(url).hostname or '') else True
                if save_raw:
                    raw_rec = {
                        'id': doc_id, 'final_url': res['final_url'],
                        'http_code': response_status, 'content_type': res['content_type'],
                        'encoding': enc,
                        'body_b64': base64.b64encode(raw_bytes).decode('ascii'),
                    }
                res['error_msg'] = None
                break

            if response_status in (429, 503):
                limiter.slow_down(2.0, 300.0, halt=(response_status == 429))
                res['status'] = 'blocked' if response_status == 429 else 'http_error'
                res['error_msg'] = f'HTTP {response_status}'
                if response_status == 429:
                    break  # Giữ blocked/429; không chạy vòng tiếp rồi đổi thành pending.
            elif response_status in PERMANENT_CODES:
                res['status'] = 'blocked' if response_status in (403, 521) else 'http_error'
                if response_status in (403, 521):
                    limiter.slow_down(1.0, 300.0, halt=True)
                res['error_msg'] = f'HTTP {response_status}'
                break
            else:
                res['status'] = 'http_error'
                res['error_msg'] = f'HTTP {response_status}'
        except (httpx.TimeoutException, httpx.PoolTimeout) as e:
            latency = time.monotonic() - t0
            res['status'] = 'timeout'; res['error_msg'] = f'{type(e).__name__}: {e}'[:200]
        except (httpx.ConnectError, httpx.RemoteProtocolError, httpx.ReadError, httpx.WriteError) as e:
            # Gộp các lỗi đứt kết nối/reset stream vào nhóm có thể tự động retry
            latency = time.monotonic() - t0
            res['status'] = 'dns_error'; res['error_msg'] = f'{type(e).__name__}: {e}'[:200]
        except Exception as e:
            latency = time.monotonic() - t0
            res['status'] = 'http_error'; res['error_msg'] = f'{type(e).__name__}: {e}'[:200]
        finally:
            limiter.release()

        retryable = (res['status'] in {'timeout', 'dns_error'}) or \
                    (res['http_code'] is not None and res['http_code'] in RETRYABLE_CODES)
        if retryable and not limiter.halted and attempt < allowed_attempts - 1:
            await asyncio.sleep(_jitter(RETRY_DELAYS[min(attempt, len(RETRY_DELAYS) - 1)]))
            continue
        break

    # ghi metrics
    limiter.record(ok=(res['status'] == 'ok'), n_bytes=res['n_bytes'], latency=latency,
                   status=res['status'], error=res['error_msg'])
    return res, raw_rec
# Chặn cả URL trực tiếp và redirect tới Long Chau trong client crawl thường.
async def guard_regular_crawler_request(request):
    host=(urlparse(str(request.url)).hostname or '').lower().removeprefix('www.')
    home_domains = {'nhathuoclongchau.com.vn', 'tamanhhospital.vn'}
    if any(host == d or host.endswith('.' + d) for d in home_domains):
        raise httpx.RequestError('Home-only domain: use the assigned home crawl and import its shards', request=request)


In [ ]:
# ---- Shard writer & status flusher (atomic, batched) ----
import zstandard as zstd
import threading

class ShardWriter:
    """Ghi shard .jsonl.zst nguyên bytes đã crawl; mỗi shard tối đa 10.000 bản ghi."""
    def __init__(self, html_dir: Path, shard_size: int = 10_000):
        self.html_dir = html_dir
        self.shard_size = shard_size
        existing = sorted(html_dir.glob('shard_*.jsonl.zst'))
        self.shard_idx = max((int(p.name.split('_')[1].split('.')[0]) for p in existing), default=-1) + 1
        # DB có thể trỏ tới shard cũ chưa đồng bộ về máy local.
        # Không tái sử dụng số shard đó khi ghi dữ liệu mới.
        database = globals().get('con')
        if database is not None:
            db_max = database.execute('SELECT coalesce(max(shard), -1) FROM crawl_status').fetchone()[0]
            self.shard_idx = max(self.shard_idx, int(db_max) + 1)
            has_imports = database.execute("SELECT count(*) FROM information_schema.tables WHERE table_name='home_import_shards'").fetchone()[0]
            if has_imports:
                reserved = database.execute('SELECT coalesce(max(shard), -1) FROM home_import_shards').fetchone()[0]
                self.shard_idx = max(self.shard_idx, int(reserved) + 1)
        self.buffer: List[Dict[str, Any]] = []
        self.cctx = zstd.ZstdCompressor(level=3, threads=-1)
        self.lock = threading.RLock()

    def add(self, rec: Dict[str, Any]) -> Optional[Path]:
        with self.lock:
            self.buffer.append(rec)
            if len(self.buffer) >= self.shard_size:
                return self.flush()
            return None

    def flush(self) -> Optional[Path]:
        with self.lock:
            if not self.buffer:
                return None
            final = self.html_dir / f'shard_{self.shard_idx:05d}.jsonl.zst'
            tmp = final.with_suffix('.jsonl.zst.tmp')
            if final.exists():
                raise FileExistsError(f'Từ chối ghi đè shard đã có: {final}')
            with open(tmp, 'wb') as f:
                with self.cctx.stream_writer(f) as writer:
                    for rec in self.buffer:
                        writer.write((json.dumps(rec, ensure_ascii=False) + '\n').encode('utf-8'))
            os.replace(tmp, final)
            n = len(self.buffer)
            self.buffer.clear()
            self.shard_idx += 1
            logger.info(f'Ghi shard {final.name} ({n} bản ghi)')
            return final


class StatusFlusher:
    """Chỉ commit status của response có raw sau khi shard chứa body đã được ghi bền vững."""
    def __init__(self, con, batch_size: int = 1_000):
        self.con = con
        self.batch_size = batch_size
        self.buffer: List[Tuple] = []
        self.pending_raw: List[Tuple] = []
        self.total = 0
        self.lock = threading.RLock()

    @staticmethod
    def _row(res: Dict[str, Any]) -> Tuple:
        return (
            res['final_url'], res['status'], res['http_code'], res['content_type'],
            res['n_bytes'], res['attempts'], res['fetched_at'], res['shard'],
            res['sha1_raw'], res['error_msg'], res['id'],
        )

    def add(self, res: Dict[str, Any], raw_pending: bool = False):
        row = self._row(res)
        with self.lock:
            if raw_pending:
                self.pending_raw.append(row)
                return
            self.buffer.append(row)

    def shard_durable(self):
        """Gọi sau khi ShardWriter.flush() đã hoàn tất thành công."""
        with self.lock:
            if self.pending_raw:
                self.buffer.extend(self.pending_raw)
                self.pending_raw.clear()

    def flush(self):
        with self.lock:
            if not self.buffer:
                return
            rows = list(self.buffer)
            self.con.execute('BEGIN TRANSACTION')
            try:
                self.con.executemany(
                    """UPDATE crawl_status
                       SET final_url=?, status=?, http_code=?, content_type=?, n_bytes=?,
                           attempts=?, fetched_at=?, shard=?, sha1_raw=?, error_msg=?
                       WHERE id=?""",
                    rows,
                )
                self.con.execute('COMMIT')
            except Exception:
                self.con.execute('ROLLBACK')
                raise
            self.total += len(rows)
            del self.buffer[:len(rows)]


In [ ]:
# ---- Orchestrator: bounded queue + graceful shutdown ----
STOP_FILE = CRAWL_DIR / 'STOP'

class GracefulStop:
    def __init__(self):
        self.flag = False
        self._prev = {}
    def install(self):
        for sig in (signal.SIGINT, signal.SIGTERM):
            try:
                self._prev[sig] = signal.getsignal(sig)
                signal.signal(sig, self._handler)
            except (ValueError, OSError):
                pass
    def restore(self):
        for sig, h in self._prev.items():
            try: signal.signal(sig, h)
            except (ValueError, OSError): pass
    def _handler(self, signum, frame):
        logger.warning(f'Nhận signal {signum}; yêu cầu dừng mềm...')
        self.flag = True
    def check(self) -> bool:
        if self.flag: return True
        if STOP_FILE.exists():
            logger.warning(f'Phát hiện {STOP_FILE.name}; dừng mềm.')
            return True
        return False

async def run_crawler_pipeline(rows_df: pd.DataFrame,
                               max_total_connections: int = 100,
                               n_workers: int = 64,
                               shard_size: int = 10_000,
                               save_interval_sec: float = 60.0) -> Dict[str, Any]:
    """Crawl với queue bounded; URL chưa xử lý sẽ còn pending để lần sau resume."""
    if save_interval_sec <= 0 or n_workers <= 0 or max_total_connections <= 0 or shard_size <= 0:
        raise ValueError('save_interval_sec, n_workers, max_total_connections and shard_size must be positive')
    rows_df = rows_df.drop_duplicates('id').copy()
    # Re-read persisted state immediately before crawling, so rerunning this cell
    # skips IDs completed by a previous run even if sample_df is stale in memory.
    con.register('_crawl_resume_ids', rows_df[['id']].drop_duplicates())
    try:
        saved = con.execute('SELECT c.id, c.status, c.attempts, c.fetched_at FROM crawl_status c JOIN _crawl_resume_ids r USING(id)').df()
    finally:
        con.unregister('_crawl_resume_ids')
    saved_by_id = saved.set_index('id').to_dict(orient='index')
    def still_queued(row):
        current = saved_by_id.get(int(row['id']))
        if current is None:
            return False
        if str(current.get('status') or '').lower() == 'pending':
            return True
        old_attempts = row.get('attempts', 0)
        current_attempts = current.get('attempts', 0)
        same_attempt = ((0 if pd.isna(current_attempts) else int(current_attempts))
                        == (0 if pd.isna(old_attempts) else int(old_attempts)))
        return (str(current.get('status') or '').lower() in {'timeout', 'dns_error', 'http_error', 'blocked'}
                and str(current.get('status') or '').lower() == str(row.get('status') or '').lower()
                and same_attempt)
    before = len(rows_df)
    mask = pd.Series([still_queued(row) for row in rows_df.to_dict(orient='records')], index=rows_df.index, dtype=bool)
    rows_df = rows_df.loc[mask].copy()
    logger.info(f'Resume before crawl: queued={len(rows_df):,}, skipped_saved_or_changed={before-len(rows_df):,}')
    manager = LimiterManager(domain_plan_dict)
    shard = ShardWriter(HTML_DIR, shard_size=shard_size)
    flusher = StatusFlusher(con, batch_size=1_000)
    stop = GracefulStop(); stop.install()
    counters = {'ok': 0, 'non_html': 0, 'fail': 0, 'stopped': 0}
    t_start = time.monotonic()
    metrics_path = REPORT_DIR / 'crawl_metrics_latest.json'
    def save_metrics():
        longchau_rows = con.execute("""
            SELECT status, http_code, COUNT(*) AS n
            FROM crawl_status
            WHERE lower(url) LIKE '%nhathuoclongchau.com.vn%'
            GROUP BY status, http_code
            ORDER BY status, http_code
        """).fetchall()
        longchau_status_counts = [
            {'status': status, 'http_code': http_code, 'count': int(n)}
            for status, http_code, n in longchau_rows
        ]
        con.register('_active_crawl_ids', rows_df[['id']].drop_duplicates())
        try:
            active_rows = con.execute("""
                SELECT cs.status, cs.http_code, COUNT(*) AS n
                FROM crawl_status cs
                JOIN _active_crawl_ids a USING (id)
                GROUP BY cs.status, cs.http_code
                ORDER BY cs.status, cs.http_code
            """).fetchall()
        finally:
            con.unregister('_active_crawl_ids')
        active_batch_status_counts = [
            {'status': status, 'http_code': http_code, 'count': int(n)}
            for status, http_code, n in active_rows
        ]
        updated_at = datetime.now(timezone.utc).isoformat()
        metrics = {'updated_at': updated_at,
                   'duration_sec': max(time.monotonic() - t_start, 0.001),
                   'processed': sum(counters.values()), **counters,
                   'urls_per_min': sum(counters.values()) * 60.0 / max(time.monotonic() - t_start, 0.001),
                   'queued': len(rows_df), 'domain_snapshots': manager.all_snapshots(),
                   'nhathuoclongchau_status_counts': longchau_status_counts,
                   'active_batch_total': len(rows_df),
                   'active_batch_status_counts': active_batch_status_counts}
        metrics_path.parent.mkdir(parents=True, exist_ok=True)
        temp = metrics_path.with_suffix('.json.tmp')
        temp.write_text(json.dumps(metrics, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
        os.replace(temp, metrics_path)
        return metrics

    # Giảm keepalive_connections và đặt keepalive_expiry=15s để tránh lỗi ngắt kết nối đột ngột
    limits = httpx.Limits(
        max_connections=max_total_connections,
        max_keepalive_connections=20,
        keepalive_expiry=15.0,
    )
    queue: asyncio.Queue = asyncio.Queue(maxsize=2_000)
    cols = list(rows_df.columns)
    progress = tqdm(total=len(rows_df), desc='Crawling', unit='url')
    last_save = t_start
    save_lock = asyncio.Lock()

    async def producer():
        try:
            for tup in rows_df.itertuples(index=False, name=None):
                if stop.check():
                    logger.info('Producer dừng; URL chưa enqueue sẽ giữ pending.')
                    break
                await queue.put(dict(zip(cols, tup)))
        finally:
            # Worker tiếp tục drain queue; sentinel không bị timeout giả khi domain chậm.
            for _ in range(n_workers):
                await queue.put(None)

    async def worker(worker_id: int):
        nonlocal last_save
        while True:
            row = await queue.get()
            try:
                if row is None:
                    return
                if stop.check():
                    counters['stopped'] += 1
                    progress.update(1)
                    continue
                counted = False
                try:
                    res, raw_rec = await fetch_with_retry(client, manager, row, shard.shard_idx)
                    if str(res.get('error_msg') or '').startswith('domain_halted:'):
                        # Chưa gửi request: giữ nguyên status/attempts đã lưu của URL.
                        counters['stopped'] += 1
                        continue
                    # 'ok' bắt buộc phải có body raw; status chỉ được commit sau shard durable.
                    if res['status'] == 'ok' and raw_rec is None:
                        raise RuntimeError('Invariant violated: status=ok without raw body')
                    # Serialize all shared writer, DB and checkpoint operations.
                    async with save_lock:
                        path = shard.add(raw_rec) if raw_rec else None
                        if raw_rec:
                            res['shard'] = int(path.name.split('_')[1].split('.')[0]) if path else shard.shard_idx
                        # Persist status for a response only after its raw body is in an atomic shard.
                        flusher.add(res, raw_pending=bool(raw_rec))
                        if raw_rec and path is not None:
                            flusher.shard_durable()
                        outcome = res['status'] if res['status'] in ('ok', 'non_html') else 'fail'
                        counters[outcome] += 1
                        counted = True
                        if time.monotonic() - last_save >= save_interval_sec:
                            if shard.flush() is not None:
                                flusher.shard_durable()
                            flusher.flush()
                            con.execute('CHECKPOINT')
                            last_save = time.monotonic()
                            saved_metrics = save_metrics()
                            logger.info(f'CRAWL_SAVED committed_statuses={flusher.total} queued={len(rows_df)} '
                                        f'active_batch_statuses={saved_metrics["active_batch_status_counts"]}')
                except Exception:
                    if counted:
                        # Lỗi lưu dữ liệu phải dừng lượt crawl, không đếm URL lần hai.
                        raise
                    counters['fail'] += 1
                    logger.exception(f'worker {worker_id} lỗi; URL sẽ còn pending nếu status chưa commit')
                finally:
                    progress.update(1)
            finally:
                queue.task_done()

    try:
        # Sử dụng http2=False để tránh lỗi RemoteProtocolError trên Cloudflare/Nginx
        async with httpx.AsyncClient(limits=limits, http2=False, verify=True, trust_env=False, event_hooks={'request':[guard_regular_crawler_request]}) as client:
            producer_task = asyncio.create_task(producer())
            workers = [asyncio.create_task(worker(i)) for i in range(n_workers)]
            tasks = [producer_task, *workers]
            try:
                await asyncio.gather(*tasks)
            except Exception:
                logger.exception('Crawler task lỗi; huỷ worker còn lại, URL chưa commit sẽ resume lần sau')
                for task in tasks:
                    if not task.done():
                        task.cancel()
                await asyncio.gather(*tasks, return_exceptions=True)
                raise
    finally:
        try:
            async with save_lock:
                final_path = shard.flush()
                if final_path is not None:
                    flusher.shard_durable()
                flusher.flush()
                con.execute('CHECKPOINT')
                save_metrics()
        finally:
            progress.close()
            stop.restore()
            if STOP_FILE.exists():
                try: STOP_FILE.unlink()
                except OSError: pass

    duration = max(time.monotonic() - t_start, 0.001)
    processed = sum(counters.values())
    metrics = {
        'duration_sec': duration,
        'processed': processed,
        'ok': counters['ok'],
        'non_html': counters['non_html'],  # Đã bổ sung để sửa lỗi KeyError: 'non_html'
        'fail': counters['fail'],
        'stopped': counters['stopped'],
        'urls_per_min': processed * 60.0 / duration,
        'domain_snapshots': manager.all_snapshots(),
    }
    logger.info(f'Hết hàng đợi của lượt crawl: {metrics["processed"]} url | ok={metrics["ok"]} '
                f'| non_html={metrics["non_html"]} | fail={metrics["fail"]} '
                f'| {metrics["urls_per_min"]:.1f} url/phút')
    save_metrics()
    logger.info(f'Metrics saved: {metrics_path}')
    return metrics
run_crawler_pipeline.review_build = '2026-10-21-review-v26'


In [ ]:
# ---- Chỉ xem trạng thái Long Châu; không reset về pending ----
# nhathuoclongchau + tamanhhospital: NT crawl bằng crawl_home.py tại máy nhà.
# Không gửi lại blocked/403 từ Colab. Hợp nhất html_home bằng cell cuối bước crawl.
display(con.execute("""
    SELECT regexp_replace(lower(split_part(url, '/', 3)), '^www[.]', '') AS domain,
           status, http_code, COUNT(*) AS n
    FROM crawl_status WHERE lower(url) LIKE '%longchau.com.vn%'
    GROUP BY domain, status, http_code ORDER BY domain, n DESC
""").df())


### Cổng kiểm tra domain (preflight)

Một khi Cloudflare trả 403/challenge cho client, **mọi** URL của domain đó cũng 403: lỗi thuộc về
client + IP, không thuộc từng URL. Vì vậy trước khi xếp hàng chục nghìn URL, mỗi domain chỉ thử
**một** URL. Domain bị chặn thì toàn bộ URL giữ `pending` (không đốt ngân sách `attempts`, không
làm bẩn `status`), và bảng `domain_gate` ghi lại bằng chứng kèm thời điểm kiểm tra.


In [ ]:
# ---- Cổng kiểm tra domain: 1 request/domain trước khi xếp hàng ----
# 20/10/2026, kiểm từ IP nhà: nhathuoclongchau.com.vn trả 403 + trang challenge cho UA crawler,
# cho UA Chrome kèm Sec-Fetch, và cho cả Chrome headless. 403 vì thế là chặn theo dấu vết client/IP,
# không theo User-Agent và không theo tốc độ: hạ rate trong notebook không sửa được lỗi này.
# Chi tiết + bằng chứng: reviews/2026-10-20_longchau_access.md
con.execute("""
CREATE TABLE IF NOT EXISTS domain_gate (
    domain     VARCHAR PRIMARY KEY,
    checked_at TIMESTAMP,
    verdict    VARCHAR,
    http_code  INTEGER,
    detail     VARCHAR,
    probe_url  VARCHAR
)
""")

GATE_MARKERS = ('cf-chl-', 'challenge-platform', 'just a moment', 'checking your browser',
                'verify you are human', 'captcha', 'robot or human')


async def probe_domain(client: httpx.AsyncClient, domain: str, probe_url: str) -> Dict[str, Any]:
    """Thử một URL: ok | challenge | blocked | http_error | unreachable."""
    rec = {'domain': domain, 'checked_at': datetime.now(timezone.utc).isoformat(),
           'verdict': 'unreachable', 'http_code': None, 'detail': None, 'probe_url': probe_url}
    try:
        resp = await client.get(probe_url, headers=REQUEST_HEADERS, follow_redirects=True, timeout=TIMEOUT)
    except Exception as e:
        rec['detail'] = f'{type(e).__name__}: {e}'[:200]
        return rec
    body = resp.content[:200_000].decode('utf-8', errors='ignore').lower()
    rec['http_code'] = resp.status_code
    if resp.headers.get('cf-mitigated') == 'challenge' or any(m in body for m in GATE_MARKERS):
        rec['verdict'], rec['detail'] = 'challenge', 'trang challenge chống bot'
    elif resp.status_code == 200:
        rec['verdict'] = 'ok'
    elif resp.status_code in (403, 429, 503):
        rec['verdict'], rec['detail'] = 'blocked', f'HTTP {resp.status_code}'
    else:
        rec['verdict'], rec['detail'] = 'http_error', f'HTTP {resp.status_code}'
    return rec


def probe_url_for(domain: str) -> Optional[str]:
    """Ưu tiên URL từng trả 200; nếu chưa có thì lấy URL đầu tiên."""
    like = f'%://{domain}/%'
    row = con.execute("SELECT url FROM crawl_status WHERE status = 'ok' AND lower(url) LIKE ? LIMIT 1",
                      [like]).fetchone()
    if row:
        return row[0]
    row = con.execute('SELECT url FROM crawl_status WHERE lower(url) LIKE ? LIMIT 1', [like]).fetchone()
    return row[0] if row else None


async def run_domain_gate(domains: List[str], delay_sec: float = 3.0) -> List[Dict[str, Any]]:
    """Tuần tự, mỗi domain 1 request, cách nhau 3 giây; ghi kết quả vào domain_gate."""
    results: List[Dict[str, Any]] = []
    async with httpx.AsyncClient(http2=False, trust_env=False, event_hooks={'request':[guard_regular_crawler_request]}, limits=httpx.Limits(max_connections=1)) as client:
        for dom in sorted(domains):
            probe = probe_url_for(dom)
            if probe is None:
                results.append({'domain': dom, 'checked_at': datetime.now(timezone.utc).isoformat(),
                                'verdict': 'no_probe', 'http_code': None,
                                'detail': 'không có URL để thử', 'probe_url': None})
                continue
            rec = await probe_domain(client, dom, probe)
            results.append(rec)
            logger.info(f'[gate] {dom}: {rec["verdict"]} (HTTP {rec["http_code"]}) {rec["detail"] or ""}')
            await asyncio.sleep(delay_sec)
    con.executemany(
        'INSERT OR REPLACE INTO domain_gate VALUES (?,?,?,?,?,?)',
        [(r['domain'], r['checked_at'], r['verdict'], r['http_code'], r['detail'], r['probe_url'])
         for r in results],
    )
    con.execute('CHECKPOINT')
    return results


In [ ]:
# ---- Chạy cổng kiểm tra cho các domain còn lỗi (1 request/domain, cách 3 giây) ----
RUN_DOMAIN_GATE = True
# Domain chạy ở nguồn khác: 120ask trên VPS, pubmed qua API, zysjonline đã có sẵn dữ liệu.
GATE_EXCLUDE = {'120ask.com', 'pubmed.ncbi.nlm.nih.gov', 'zysjonline.com',
                'nhathuoclongchau.com.vn', 'tamanhhospital.vn',
                'pmc-ecm-healthblog.beta.pharmacity.io', 'vov.vn', 'article.iiyi.com', 'bingli.iiyi.com'}

candidates = [d for d, in con.execute("""
    SELECT DISTINCT regexp_replace(regexp_extract(url, 'https?://([^/]+)', 1), '^www\\.', '')
    FROM crawl_status
    WHERE status IN ('pending', 'blocked', 'http_error', 'timeout', 'dns_error')
""").fetchall() if d and not any(d == x or d.endswith('.' + x) for x in GATE_EXCLUDE)]

if RUN_DOMAIN_GATE and candidates:
    gate_results = asyncio.run(run_domain_gate(candidates))
else:
    gate_results = []
    print(f'Cổng domain tắt hoặc không có domain cần thử (candidates={len(candidates)}).')

gate_df = con.execute("""
    SELECT verdict, COUNT(*) AS n_domain,
           string_agg(domain, ', ') FILTER (WHERE verdict <> 'ok') AS domains
    FROM domain_gate GROUP BY verdict ORDER BY n_domain DESC
""").df()
print('Kết quả cổng domain (toàn bộ lần kiểm tra gần nhất):')
display(gate_df)


In [ ]:
# ---- iiyi: một probe/domain mỗi 24 giờ, chỉ xếp lại URL 521 khi probe thành công ----
async def retry_iiyi_daily():
    results = []
    for domain in ('article.iiyi.com', 'bingli.iiyi.com'):
        count = con.execute("""SELECT count(*) FROM crawl_status
            WHERE regexp_replace(lower(split_part(url, '/', 3)), '^www[.]', '')=?
              AND http_code=521 AND status<>'ok'""", [domain]).fetchone()[0]
        if not count:
            continue
        last = con.execute('SELECT checked_at FROM domain_gate WHERE domain=?', [domain]).fetchone()
        now = pd.Timestamp.now(tz='UTC')
        previous = pd.to_datetime(last[0], utc=True) if last and last[0] is not None else None
        if previous is not None and now-previous < pd.Timedelta(hours=24):
            logger.info(f'{domain}: đã probe trong 24 giờ qua, không gửi lại HTTP 521.')
            continue
        rows = await run_domain_gate([domain], delay_sec=3.0)
        results.extend(rows)
        if rows and rows[0]['verdict']=='ok':
            con.execute("""UPDATE crawl_status SET status='pending',attempts=0,http_code=NULL,
                error_msg='iiyi server recovered after daily probe'
                WHERE regexp_replace(lower(split_part(url, '/', 3)), '^www[.]', '')=?
                  AND http_code=521 AND status<>'ok'""", [domain])
            con.execute('CHECKPOINT')
            logger.info(f'{domain}: server đã hồi phục, xếp lại {count:,} URL 521.')
        else:
            logger.warning(f'{domain}: chưa hồi phục, giữ nguyên trạng thái; thử lại sau 24 giờ.')
    return results

RUN_IIYI_DAILY_PROBE = True
if RUN_IIYI_DAILY_PROBE:
    iiyi_daily_results = asyncio.run(retry_iiyi_daily())


In [ ]:
# ---- Chọn tập URL để crawl (pilot, pilot2 hoặc full; resume theo lỗi tạm thời) ----
RUN_CRAWL = True             # chạy crawler HTML khi thực thi cell run; PubMed ưu tiên ở phía trên
CRAWL_MODE = 'full'        # pilot2 = đúng ID trong pilot2_sample.csv; 'pilot' | 'full'
FULL_CRAWL_APPROVED = True   # review 2026-10-20_v3_sample2_rerun2: mẫu v26 đã đạt
N_PILOT = 10_000
N_WORKERS = 128              # tăng song song giữa các domain; limiter từng domain vẫn giữ nguyên
SKIP_DOMS = {'pubmed.ncbi.nlm.nih.gov', 'zysjonline.com',
             'nhathuoclongchau.com.vn', 'tamanhhospital.vn',
             'pmc-ecm-healthblog.beta.pharmacity.io', 'vov.vn'}
# Hai domain NT crawl ở nhà không được probe hoặc xếp hàng từ Colab.
# Long Châu chạy ở nhà bằng file longchau_ranked_tv.csv đã chia theo vị trí lẻ; không chia theo id % 2.
DOMAIN_GATE_ONLY = {'nhathuoclongchau.com.vn', 'tamanhhospital.vn'}
ALLOW_CLOUD_DOMAIN = False   # 120ask chạy riêng trên VPS theo đặc tả

# 429/403/challenge không tự retry: cần rà soát quyền truy cập trước lần chạy tiếp theo.
# attempts = tổng request đã thực hiện cho URL; nội bộ cho phép tối đa 3 retry cho lỗi tạm thời.
MAX_TOTAL_ATTEMPTS = len(RETRY_DELAYS) + 1
# Reset only old ok verification responses, once; blocked retries are not reset forever.
verify_rows = con.execute("""
    SELECT * FROM crawl_status
    WHERE status='ok' AND lower(coalesce(final_url, '')) LIKE '%verify.html%'
      AND (lower(split_part(url, '/', 3))='39.net'
           OR lower(split_part(url, '/', 3)) LIKE '%.39.net')
""").df()
if not verify_rows.empty:
    audit = REPORT_DIR / ('39net_verify_reset_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.csv')
    audit.parent.mkdir(parents=True, exist_ok=True)
    verify_rows.to_csv(audit, index=False, encoding='utf-8-sig')
    con.register('_verify_reset_ids', verify_rows[['id']])
    try:
        con.execute("""UPDATE crawl_status SET status='pending', attempts=0,
            shard=NULL, sha1_raw=NULL, error_msg='reset old verification response'
            WHERE id IN (SELECT id FROM _verify_reset_ids) AND status='ok'""")
    finally:
        con.unregister('_verify_reset_ids')
    logger.info(f'39.net: reset {len(verify_rows):,} old verify IDs (review expected 9,027); audit={audit}')

pending_df = con.execute("""
    SELECT id, url, lang_expected, status, http_code, attempts
    FROM crawl_status
    WHERE (status = 'pending' AND COALESCE(attempts, 0) < ?)
       OR (status IN ('timeout', 'dns_error') AND COALESCE(attempts, 0) < ?)
       OR (status = 'http_error' AND http_code BETWEEN 500 AND 599
           AND http_code != 521 AND COALESCE(attempts, 0) < ?)
       OR (status = 'http_error' AND http_code IS NULL
           AND regexp_matches(coalesce(error_msg,''), 'StreamReset|RemoteProtocolError|StreamClosedError')
           AND COALESCE(attempts, 0) < ?)
""", [MAX_TOTAL_ATTEMPTS] * 4).df()

pending_df['domain'] = pending_df['url'].apply(lambda u: urlparse(u).netloc.lower().removeprefix('www.'))
def excluded_domain(domain, excluded):
    return any(domain == item or domain.endswith('.' + item) for item in excluded)

pending_df = pending_df[~pending_df['domain'].map(lambda d: excluded_domain(d, SKIP_DOMS))]
pending_df = pending_df[~pending_df['domain'].str.endswith('.zysjonline.com')]
if not ALLOW_CLOUD_DOMAIN:
    pending_df = pending_df[~pending_df['domain'].str.endswith('120ask.com')]

# Cổng domain: bỏ URL của domain đã bị chặn/challenge ở lần kiểm tra gần nhất, giữ nguyên pending.
try:
    gate_rows = con.execute('SELECT domain, verdict FROM domain_gate').fetchall()
except Exception:
    gate_rows = []
    logger.warning('Chưa có bảng domain_gate: chạy cell cổng domain phía trên trước.')
gate_blocked = {d for d, v in gate_rows if v != 'ok'}
gate_blocked |= DOMAIN_GATE_ONLY
# Một subdomain 39.net bị chặn thì dừng cả họ domain dùng chung ngân sách.
if any(d == '39.net' or d.endswith('.39.net') for d in gate_blocked):
    gate_blocked.add('39.net')
gate_mask = pending_df['domain'].map(lambda d: excluded_domain(d, gate_blocked)).astype(bool)
n_gate = int(gate_mask.sum())
pending_df = pending_df[~gate_mask]
if n_gate:
    logger.warning(f'Cổng domain: giữ pending {n_gate:,} URL của {len(gate_blocked)} domain '
                   f'chưa truy cập được: {sorted(gate_blocked)}')

n_retry = int((pending_df['status'] != 'pending').sum())
logger.info(f'Resume: {len(pending_df):,} URL (pending={(pending_df["status"]=="pending").sum():,}, retry={n_retry:,})')

if CRAWL_MODE == 'pilot':
    strata = list(pending_df.groupby(['lang_expected', 'domain'], dropna=False, sort=True))
    min_rows = pd.concat([g.sample(min(30, len(g)), random_state=42) for _, g in strata]) if strata else pending_df.head(0)
    if len(min_rows) > N_PILOT:
        raise ValueError('Pilot quota vượt N_PILOT; giảm số URL/domain tối thiểu.')
    remainder = pending_df.drop(index=min_rows.index)
    extra = min(N_PILOT - len(min_rows), len(remainder))
    sample_df = pd.concat([min_rows, remainder.sample(n=extra, random_state=42)]) if extra > 0 else min_rows
elif CRAWL_MODE == 'pilot2':
    pilot2_csv = next((x for x in [CRAWL_DIR/'pilot2_sample.csv', DATA_ROOT/'pilot2_sample.csv', Path('crawl/pilot2_sample.csv')] if x.is_file()), None)
    if pilot2_csv is None:
        raise FileNotFoundError(f'pilot2_sample.csv not found in {CRAWL_DIR} or {DATA_ROOT}')
    pilot2 = pd.read_csv(pilot2_csv, usecols=lambda c: c in ('id','url'))
    if not {'id','url'}.issubset(pilot2.columns):
        raise ValueError(f'{pilot2_csv} must contain id,url columns')
    pilot2['id'] = pd.to_numeric(pilot2['id'], errors='raise').astype('int64')
    if pilot2['id'].duplicated().any():
        raise ValueError('pilot2_sample.csv contains duplicate IDs')
    pilot2_ids = set(pilot2['id'])
    if len(pilot2_ids) != 40000:
        logger.warning(f'pilot2_sample.csv has {len(pilot2_ids):,} IDs; expected 40,000.')
    source_urls = dict(zip(df_links['id'].astype('int64'), df_links['url'].astype(str)))
    unknown_ids = pilot2_ids - set(source_urls)
    if unknown_ids:
        raise ValueError(f'{len(unknown_ids)} pilot2 IDs are missing from the source URL table')
    mismatched_urls = [int(r.id) for r in pilot2.itertuples(index=False) if str(r.url) != source_urls[int(r.id)]]
    if mismatched_urls:
        raise ValueError(f'pilot2 URL differs from source URL table for IDs: {mismatched_urls[:10]}')

    # Requeue successful rows only when their recorded raw shard is missing from this Drive.
    existing_shards = {int(m.group(1)) for f in HTML_DIR.glob('shard_*.jsonl.zst') if (m := re.search(r'shard_(\d+)', f.name))}
    con.register('_pilot2_ids', pd.DataFrame({'id': sorted(pilot2_ids)}))
    try:
        pilot_status = con.execute('SELECT c.id, c.status, c.shard FROM crawl_status c JOIN _pilot2_ids p USING (id)').df()
    finally:
        con.unregister('_pilot2_ids')
    stale_ids = [int(r.id) for r in pilot_status.itertuples(index=False)
                 if str(r.status).lower() in {'ok','non_html'} and (pd.isna(r.shard) or int(r.shard) not in existing_shards)]
    # Loại domain chạy ở nguồn khác TRƯỚC khi sửa DB; không reset Long Châu về pending.
    stale_ids = [rid for rid in stale_ids
                 if not excluded_domain((urlparse(source_urls[rid]).hostname or '').lower().removeprefix('www.'), SKIP_DOMS | gate_blocked)
                 and (ALLOW_CLOUD_DOMAIN or not excluded_domain((urlparse(source_urls[rid]).hostname or '').lower(), {'120ask.com'}))]
    if stale_ids:
        con.executemany("UPDATE crawl_status SET final_url=NULL,status='pending',http_code=NULL,content_type=NULL,n_bytes=NULL,attempts=0,fetched_at=NULL,shard=NULL,sha1_raw=NULL,error_msg=NULL WHERE id=?", [(x,) for x in stale_ids])
        stale_rows = df_links[df_links['id'].isin(stale_ids)][['id','url','lang_expected']].copy()
        stale_rows['status']='pending'; stale_rows['http_code']=None; stale_rows['attempts']=0
        stale_rows['domain']=stale_rows['url'].apply(lambda u: urlparse(u).netloc.lower().removeprefix('www.'))
        stale_rows = stale_rows[~stale_rows['domain'].map(lambda d: excluded_domain(d, SKIP_DOMS | gate_blocked))]
        stale_rows = stale_rows[~stale_rows['domain'].str.endswith('.zysjonline.com')]
        if not ALLOW_CLOUD_DOMAIN:
            stale_rows = stale_rows[~stale_rows['domain'].str.endswith('120ask.com')]
        pending_df = pd.concat([pending_df, stale_rows], ignore_index=True).drop_duplicates('id', keep='last')
        logger.warning(f'Requeued {len(stale_rows):,} pilot2 rows marked successful but missing their raw shard.')
    sample_df = pending_df[pending_df['id'].isin(pilot2_ids)].copy()
    logger.info(f'Pilot2 IDs={len(pilot2_ids):,}; queued={len(sample_df):,}; existing_shards={len(existing_shards)}')
else:
    if CRAWL_MODE != 'full': raise ValueError(f'Unknown CRAWL_MODE: {CRAWL_MODE}')
    if RUN_CRAWL and not FULL_CRAWL_APPROVED:
        raise RuntimeError('Full crawl đang khóa. Bật FULL_CRAWL_APPROVED có chủ ý sau khi rà soát cấu hình và nguồn đặc biệt.')
    sample_df = pending_df

sample_df = sample_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
logger.info(f'Tập URL sẽ crawl: {len(sample_df):,} ({CRAWL_MODE})')
print(f'Tổng URL chọn: {len(sample_df):,} ({CRAWL_MODE})')
sample_df.head()

In [ ]:
# ---- Chạy crawler & báo cáo pilot ----
metrics_path = REPORT_DIR / 'crawl_metrics_latest.json'
CRAWL_METRICS = json.loads(metrics_path.read_text(encoding='utf-8')) if metrics_path.is_file() else None
if RUN_CRAWL:
    # Colab có thể còn giữ hàm cũ dù file notebook đã được cập nhật.
    import types
    worker_code = next((c for c in run_crawler_pipeline.__code__.co_consts
                        if isinstance(c, types.CodeType) and c.co_name == 'worker'), None)
    if (getattr(run_crawler_pipeline, 'review_build', None) != '2026-10-21-review-v26'
            or worker_code is None or 'last_save' not in worker_code.co_freevars):
        raise RuntimeError('Đang giữ cell crawler cũ. Chạy lại cell Orchestrator: run_crawler_pipeline trước.')
    CRAWL_METRICS = asyncio.run(run_crawler_pipeline(
        sample_df, max_total_connections=150, n_workers=N_WORKERS, shard_size=10_000, save_interval_sec=60.0,
    ))
else:
    print('RUN_CRAWL=False; bỏ qua crawl. Đặt True sau khi duyệt.')


In [ ]:
# ---- Báo cáo pilot_crawl.md ----
if RUN_CRAWL and CRAWL_METRICS:
    con.register('pilot_selection', sample_df[['id']])
    pilot_stats = con.execute("""
        SELECT cs.status, COUNT(*) AS n_urls,
               ROUND(AVG(cs.n_bytes)/1024.0, 1) AS avg_kb,
               ROUND(AVG(cs.attempts), 2)       AS avg_attempts
        FROM crawl_status cs INNER JOIN pilot_selection s USING (id)
        GROUP BY cs.status ORDER BY n_urls DESC
    """).df()
    pilot_domain = con.execute("""
        SELECT cs.lang_expected,
               regexp_replace(regexp_extract(cs.url, 'https?://([^/]+)', 1), '^www\\.', '') AS domain,
               COUNT(*) AS total,
               SUM((cs.status='ok')::INT)  AS ok_count,
               SUM((cs.status!='ok')::INT) AS fail_count,
               ROUND(100.0*SUM((cs.status='ok')::INT)/COUNT(*), 1) AS ok_pct,
               ROUND(AVG(cs.n_bytes)/1024.0, 1) AS avg_kb
        FROM crawl_status cs INNER JOIN pilot_selection s USING (id)
        GROUP BY cs.lang_expected, domain
        ORDER BY fail_count DESC, total DESC
    """).df()
    con.unregister('pilot_selection')

    m = CRAWL_METRICS
    lines = [
        '# Pilot Crawl Report',
        f'Generated: {datetime.now().isoformat(timespec="seconds")}',
        f'Duration: {m["duration_sec"]:.1f}s  |  Speed: {m["urls_per_min"]:.1f} url/phút',
        f'Processed={m["processed"]:,}  ok={m["ok"]:,}  fail={m["fail"]:,}  stopped={m["stopped"]:,}',
        '',
        '## Status distribution',
        pilot_stats.to_markdown(index=False),
        '',
        '## Per-language / per-domain',
        pilot_domain.to_markdown(index=False),
        '',
        '## Per-domain limiter metrics',
        pd.DataFrame(m['domain_snapshots']).to_markdown(index=False),
    ]
    (REPORT_DIR / 'pilot_crawl.md').write_text('\n'.join(lines), encoding='utf-8')
    logger.info(f'Đã ghi {REPORT_DIR / "pilot_crawl.md"}')
    display(pilot_stats)
    display(pilot_domain.head(30))
else:
    print('Chưa chạy crawler → không tạo báo cáo pilot.')


### VOV: thử Playwright Chrome headless

VOV được tách khỏi httpx vì review ghi 403 ở mọi IP. Bật `RUN_VOV_BROWSER=True` để thử một URL trước; nếu được mới đặt `VOV_MAX_URLS=None` để thu hết, nhịp tối đa một trang/2 giây. Probe thất bại sẽ dừng ngay và xuất `reports/vov_browser_probe.csv` để báo NT. Cell dùng [Playwright async API](https://playwright.dev/python/docs/api/class-page) và [Chrome channel](https://playwright.dev/python/docs/browsers#google-chrome--microsoft-edge).


In [ ]:
# ---- VOV: thử Chrome headless; dừng domain ngay khi probe thất bại ----
async def crawl_vov_headless(max_urls=1, delay_sec=2.0, channel='chrome'):
    if delay_sec < 2.0 or (max_urls is not None and max_urls <= 0):
        raise ValueError('VOV cần ít nhất 2 giây/trang; max_urls là số dương hoặc None.')
    from playwright.async_api import async_playwright
    rows = con.execute("""SELECT id,url,attempts FROM crawl_status
        WHERE regexp_replace(lower(split_part(url, '/', 3)), '^www[.]', '')='vov.vn'
          AND status NOT IN ('ok','non_html') AND coalesce(http_code,0) NOT IN (404,410)
        ORDER BY id""").df()
    if max_urls is not None:
        rows = rows.head(int(max_urls))
    shard = ShardWriter(HTML_DIR, shard_size=10000)
    flusher = StatusFlusher(con)
    report = []
    last_save = time.monotonic()
    def save():
        if shard.flush() is not None:
            flusher.shard_durable()
        flusher.flush()
        con.execute('CHECKPOINT')
    try:
        async with async_playwright() as playwright:
            browser = await playwright.chromium.launch(headless=True, channel=channel)
            try:
                page = await browser.new_page()
                for row in rows.itertuples(index=False):
                    if STOP_FILE.exists():
                        break
                    url = row.url
                    started = time.monotonic()
                    raw, code, final_url, content_type, error = b'', None, url, '', None
                    try:
                        response = await page.goto(url, wait_until='domcontentloaded', timeout=60000)
                        final_url = page.url
                        if response is not None:
                            code = response.status
                            content_type = response.headers.get('content-type', '')
                            raw = await response.body()
                        low = raw[:200000].decode('utf-8', errors='ignore').lower()
                        challenge = ('verify.html' in final_url.lower()
                                     or any(marker in low for marker in GATE_MARKERS))
                        html = not content_type or 'html' in content_type.lower()
                        state = ('blocked' if challenge or code in (401,403,429) else
                                 'ok' if code==200 and html and raw else 'http_error')
                        if state!='ok':
                            error = f'VOV headless probe failed: HTTP {code}, challenge={challenge}'
                    except Exception as exc:
                        state, error = 'http_error', f'{type(exc).__name__}: {exc}'[:300]
                    res = dict(id=int(row.id),final_url=final_url,status=state,http_code=code,
                        content_type=content_type,n_bytes=len(raw),attempts=(0 if pd.isna(row.attempts) else int(row.attempts))+1,
                        fetched_at=datetime.now(timezone.utc).isoformat(),shard=None,
                        sha1_raw=hashlib.sha1(raw).hexdigest() if state=='ok' else None,error_msg=error)
                    if state=='ok':
                        record = dict(id=int(row.id),final_url=final_url,http_code=code,content_type=content_type,
                                      encoding=detect_encoding(raw,content_type),body_b64=base64.b64encode(raw).decode('ascii'))
                        path = shard.add(record)
                        res['shard'] = int(path.name.split('_')[1].split('.')[0]) if path else shard.shard_idx
                    flusher.add(res,raw_pending=(state=='ok'))
                    if state=='ok' and path is not None:
                        flusher.shard_durable()
                    report.append(dict(id=int(row.id),url=url,final_url=final_url,status=state,http_code=code,n_bytes=len(raw),error_msg=error))
                    if state!='ok':
                        logger.warning(f'VOV: {error}; dừng domain và gửi vov_browser_probe.csv cho NT.')
                        break
                    if time.monotonic()-last_save>=60:
                        save()
                        last_save = time.monotonic()
                    await asyncio.sleep(max(0.0,delay_sec-(time.monotonic()-started)))
            finally:
                await browser.close()
    finally:
        save()
        REPORT_DIR.mkdir(parents=True,exist_ok=True)
        pd.DataFrame(report,columns=['id','url','final_url','status','http_code','n_bytes','error_msg']).to_csv(
            REPORT_DIR/'vov_browser_probe.csv',index=False,encoding='utf-8-sig')
    return pd.DataFrame(report)

# Bật cell khi sẵn sàng thử; mặc định chỉ probe một URL, không tự thu 9.784 trang.
RUN_VOV_BROWSER = False
VOV_MAX_URLS = 1  # sau khi probe thành công: đặt None để chạy hết vov còn thiếu
if RUN_VOV_BROWSER:
    import importlib.util, subprocess
    if importlib.util.find_spec('playwright') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','playwright'])
    # Linux/Colab: tải Chrome nếu chưa có. Máy Windows dùng Chrome đã cài.
    if COLAB:
        subprocess.check_call([sys.executable,'-m','playwright','install','--with-deps','chrome'])
    vov_browser_report = asyncio.run(crawl_vov_headless(max_urls=VOV_MAX_URLS))
    display(vov_browser_report)


### Hợp nhất shard crawl ở nhà của NT và TV

Đồng bộ NT vào `crawl/html_home_nt/`, TV vào `crawl/html_home_tv/`, gồm `status.csv` và thư mục `html/` hoặc các shard ở cùng cấp. Cell nhận cả `home_shard_*` và `tv_shard_*`, đổi sang số shard tiếp theo, kiểm SHA-1/body và lấy dòng status hợp lệ cuối cùng. Chạy lại không nhập trùng; status mới hơn trong DB được giữ lại.


In [ ]:
# ---- Hợp nhất crawl/html_home: chạy khi crawler đã dừng ----
def merge_home_crawl(home_dir=None):
    import csv
    import io
    import uuid
    import zstandard as zstd

    if home_dir is None:
        homes = [CRAWL_DIR / name for name in ('html_home_nt', 'html_home_tv', 'html_home')
                 if (CRAWL_DIR / name / 'status.csv').is_file()]
        if not homes:
            raise FileNotFoundError('Chưa có status.csv trong crawl/html_home_nt, html_home_tv hoặc html_home')
        return pd.concat([merge_home_crawl(home) for home in homes], ignore_index=True)
    home = Path(home_dir)
    status_file = home / 'status.csv'
    if not status_file.is_file():
        raise FileNotFoundError(status_file)
    # Dòng có status cuối cùng thắng; dòng bị cắt không được ghi đè dòng hợp lệ trước.
    last = {}
    skipped = 0
    with status_file.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            if (not row.get('status') or not row.get('id') or None in row
                    or any(value is None for value in row.values())):
                skipped += 1
                continue
            try:
                doc_id = int(row['id'])
            except (ValueError, TypeError):
                skipped += 1
                continue
            last[doc_id] = row
    if not last:
        raise ValueError('status.csv không có dòng hợp lệ')
    ids = pd.DataFrame({'id': list(last)})
    con.register('_home_ids', ids)
    try:
        known = con.execute('SELECT cs.id,cs.url FROM crawl_status cs JOIN _home_ids h USING(id)').df()
    finally:
        con.unregister('_home_ids')
    url_by_id = dict(zip(known['id'], known['url']))
    if set(last) - set(url_by_id):
        raise ValueError('Home status có ID không tồn tại trong corpus; dừng hợp nhất')
    allowed = {'nhathuoclongchau.com.vn', 'tamanhhospital.vn'}
    for doc_id, row in last.items():
        host = (urlparse(url_by_id[doc_id]).hostname or '').lower().removeprefix('www.')
        if host not in allowed or (row.get('url') and row['url'] != url_by_id[doc_id]):
            raise ValueError(f'Home ID/URL không khớp hai domain đã phân công: {doc_id}')

    con.execute('''CREATE TABLE IF NOT EXISTS home_import_shards (
        source VARCHAR PRIMARY KEY, sha256 VARCHAR, shard INTEGER UNIQUE)''')
    mapping = {}
    sources = sorted({r.get('shard_file') for r in last.values() if r.get('shard_file')})
    HTML_DIR.mkdir(parents=True, exist_ok=True)
    existing = [int(m.group(1)) for p in HTML_DIR.glob('shard_*.jsonl.zst')
                if (m := re.fullmatch(r'shard_(\d+)\.jsonl\.zst', p.name))]
    reserved = con.execute('SELECT coalesce(max(shard), -1) FROM home_import_shards').fetchone()[0]
    db_max = con.execute('SELECT coalesce(max(shard), -1) FROM crawl_status').fetchone()[0]
    next_shard = max(existing + [int(reserved), int(db_max), -1]) + 1
    for name in sources:
        if not re.fullmatch(r'(?:home|tv)_shard_\d+\.jsonl\.zst', name):
            raise ValueError(f'Tên home shard không hợp lệ: {name}')
        src = next((p for p in (home / name, home / 'html' / name) if p.is_file()), None)
        if src is None:
            raise FileNotFoundError(f'Chưa đồng bộ đủ home shard: {name}')
        def file_sha(path):
            with path.open('rb') as handle:
                return hashlib.file_digest(handle, 'sha256').hexdigest()
        digest = file_sha(src)
        key = str(src.resolve())
        saved = con.execute('SELECT sha256,shard FROM home_import_shards WHERE source=?', [key]).fetchone()
        if saved and saved[0] != digest:
            raise ValueError(f'Home shard đã thay đổi sau lần nhập trước: {name}')
        if saved:
            idx = int(saved[1])
        else:
            idx = next_shard
            next_shard += 1
            # Giữ chỗ trước copy: chạy lại sau crash sẽ dùng đúng số shard này.
            con.execute('INSERT INTO home_import_shards VALUES (?,?,?)', [key, digest, idx])
            con.execute('CHECKPOINT')
        target = HTML_DIR / f'shard_{idx:05d}.jsonl.zst'
        if target.exists():
            if file_sha(target) != digest:
                raise ValueError(f'Shard đích khác nội dung đã nhập: {target}')
        else:
            temp = target.with_name(target.name + '.' + uuid.uuid4().hex + '.tmp')
            shutil.copyfile(src, temp)
            if file_sha(temp) != digest:
                raise IOError(f'Copy home shard không khớp SHA256: {src}')
            os.replace(temp, target)
        mapping[name] = idx

    # Kiểm đủ body và SHA-1 trong shard đã đóng trước khi nâng DB thành ok.
    bodies = {}
    required_keys = {'id', 'final_url', 'http_code', 'content_type', 'encoding', 'body_b64'}
    for name, idx in mapping.items():
        path = HTML_DIR / f'shard_{idx:05d}.jsonl.zst'
        with path.open('rb') as compressed, zstd.ZstdDecompressor().stream_reader(compressed) as reader, io.TextIOWrapper(reader, encoding='utf-8') as lines:
            for line in lines:
                rec = json.loads(line)
                if not required_keys.issubset(rec):
                    raise ValueError(f'Home shard thiếu trường: {name}')
                doc_id = int(rec['id'])
                if doc_id not in last or last[doc_id].get('shard_file') != name:
                    continue
                raw = base64.b64decode(rec['body_b64'], validate=True)
                bodies[doc_id] = (hashlib.sha1(raw).hexdigest(), len(raw), int(rec['http_code']))
    updates, audit = [], []
    for doc_id, row in last.items():
        state = row['status'].strip().lower()
        raw = bodies.get(doc_id)
        name = row.get('shard_file')
        if state == 'ok' and (not name or not raw or not raw[1] or raw[2] != 200):
            raise ValueError(f'Home status=ok không có body HTTP 200 hợp lệ: {doc_id}')
        if name and (not raw or raw[0] != row.get('sha1_raw', '').lower()):
            raise ValueError(f'Home SHA-1 không khớp status/shard: {doc_id}')
        def number(key):
            value = row.get(key)
            return int(value) if value else None
        fetched = pd.to_datetime(row.get('fetched_at'), errors='coerce', utc=True)
        if pd.isna(fetched):
            raise ValueError(f'Home fetched_at thiếu hoặc không hợp lệ: {doc_id}')
        shard_id = mapping.get(name)
        updates.append((row.get('final_url') or row.get('url'), state, number('http_code'),
                        row.get('content_type') or None, number('n_bytes'), number('attempts'),
                        fetched.to_pydatetime(), shard_id, row.get('sha1_raw') or None,
                        row.get('error_msg') or None, doc_id, fetched.to_pydatetime()))
        audit.append({'id': doc_id, 'status': state, 'source_shard': name, 'shard': shard_id})
    con.execute('BEGIN TRANSACTION')
    try:
        con.executemany('''UPDATE crawl_status SET final_url=?,status=?,http_code=?,content_type=?,
            n_bytes=?,attempts=?,fetched_at=?,shard=?,sha1_raw=?,error_msg=?
            WHERE id=? AND (fetched_at IS NULL OR fetched_at<=?)''', updates)
        con.execute('COMMIT')
    except Exception:
        con.execute('ROLLBACK')
        raise
    con.execute('CHECKPOINT')
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    audit_path = REPORT_DIR / ('home_merge_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.csv')
    pd.DataFrame(audit).to_csv(audit_path, index=False, encoding='utf-8-sig')
    print(f'Home: {len(last):,} ID hợp lệ; {skipped} dòng bị cắt bỏ qua; {len(mapping)} shard; audit={audit_path}')
    return pd.DataFrame(audit)

# Chạy sau khi đồng bộ xong html_home_nt/ và/hoặc html_home_tv/; không nhập khi job đang ghi.
RUN_HOME_MERGE = False
if RUN_HOME_MERGE:
    home_merge_audit = merge_home_crawl()


In [ ]:
# ---- Trạng thái toàn corpus theo domain: hết queue không có nghĩa đủ corpus ----
# Trong kernel crawler dùng connection đang mở. Kernel khác: snapshot DB + WAL rồi mở read_only.
def report_crawl_domains(connection):
    frame = connection.execute("""
        SELECT regexp_replace(lower(split_part(url, '/', 3)), '^www[.]', '') AS domain,
               count(*) AS total, sum((status='ok')::int) AS ok,
               sum((coalesce(status, '')<>'ok')::int) AS not_ok,
               string_agg(DISTINCT status || coalesce(':' || CAST(http_code AS INT), ''), ' ') AS statuses
        FROM crawl_status GROUP BY 1 ORDER BY not_ok DESC
    """).df()
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    path = REPORT_DIR / ('crawl_status_by_domain_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.csv')
    frame.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"Corpus: {frame['ok'].sum():,}/{frame['total'].sum():,} ok; report={path}")
    display(frame)
    return frame

def snapshot_crawl_domains(status_db=None):
    import tempfile
    database = Path(status_db) if status_db else Path(globals().get('DB_PATH', CRAWL_DIR/'status.duckdb'))
    with tempfile.TemporaryDirectory(prefix='crawl_domain_status_') as folder:
        snapshot = Path(folder)/'status.duckdb'
        try:
            shutil.copy2(database,snapshot)
            wal = Path(str(database)+'.wal')
            if wal.is_file():
                shutil.copy2(wal,Path(str(snapshot)+'.wal'))
        except PermissionError as exc:
            raise RuntimeError('Windows đang khóa status.duckdb. Dừng crawler và đóng connection ở kernel ghi trước khi lấy snapshot.') from exc
        reader = duckdb.connect(str(snapshot),read_only=True)
        try:
            return report_crawl_domains(reader)
        finally:
            reader.close()

# Cùng kernel crawler: dùng connection hiện có sau checkpoint. Kernel khác gọi snapshot_crawl_domains().
if 'con' in globals():
    con.execute('CHECKPOINT')
    domain_status = report_crawl_domains(con)
else:
    domain_status = snapshot_crawl_domains()


---
## Bước 3 — Clean & trích xuất (T1.2)

Tuân thủ `DAC_TA_LAM_SACH_v1.md`:
1. Nguyên văn – không sửa chính tả/dấu câu/Unicode.
2. Whitespace: chỉ gộp; HTML entity do parser DOM giải mã 1 lần.
3. Bảng: `<tr>` → 1 block `table`, ô nối bằng `\t`.
4. List: mỗi `<li>` → 1 block, giữ `level`.
5. QA: phân biệt `qa_q`/`qa_a` trên site hỏi đáp.
6. Text = blocks nối `\n\n`; `text[char_start:char_end] == block.text` 100%.
7. Metadata: title = heading đầu; ngày ISO; category từ breadcrumb.
8. Không lọc chủ đề; chỉ loại thành phần rác theo đặc tả.


In [ ]:
# ---- Bộ công cụ làm sạch ----
try:
    from selectolax.lexbor import LexborHTMLParser as HTMLParser
except ImportError:
    from selectolax.parser import HTMLParser
from html import unescape as html_unescape

DOMAIN_LANG_MAP: Dict[str, str] = {}
try:
    df_plan = pd.read_csv('crawl/domain_plan.csv')
    for _, r in df_plan.iterrows():
        dom = str(r['domain']).strip().lower().removeprefix('www.')
        DOMAIN_LANG_MAP[dom] = str(r['lang']).strip().lower()
except Exception as e:
    logger.warning(f'Không đọc được domain_plan.csv: {e}')

CONTAINER_HINTS: Dict[str, Dict[str, Any]] = {}
TITLE_HINTS: Dict[str, str] = {}
try:
    df_hints = pd.read_csv('crawl/container_hints.csv')
    for _, r in df_hints.iterrows():
        dom = str(r['domain']).strip().lower()
        main_c = str(r['main_container']).strip() if pd.notna(r['main_container']) else ''
        also_c = str(r['also']).strip() if pd.notna(r['also']) else ''
        title_c = str(r['title']).strip() if pd.notna(r['title']) else ''
        also_sels = [re.sub(r'\(.*?\)', '', p).strip() for p in also_c.split(';')] if also_c else []
        CONTAINER_HINTS[dom] = {'main': main_c, 'also': [s for s in also_sels if s]}
        if title_c: TITLE_HINTS[dom] = title_c
    logger.info(f'Loaded {len(CONTAINER_HINTS)} container hints, {len(DOMAIN_LANG_MAP)} domain langs')
except Exception as e:
    logger.warning(f'Không đọc được container_hints.csv: {e}')

def normalize_ws(text: str) -> str:
    if not text: return ''
    return re.sub(r'\s+', ' ', text).strip()

def parse_published_date(value: Any) -> Optional[str]:
    if value is None: return None
    raw = normalize_ws(str(value))
    if not raw: return None
    for cand in (raw[:10], raw):
        for fmt in ('%Y-%m-%d', '%Y/%m/%d', '%d/%m/%Y', '%m/%d/%Y', '%B %d, %Y', '%b %d, %Y'):
            try: return datetime.strptime(cand, fmt).date().isoformat()
            except ValueError: pass
    m = re.fullmatch(r'(\d{4})年(\d{1,2})月(\d{1,2})日?', raw)
    if m:
        try: return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3))).date().isoformat()
        except ValueError: return None
    return None

SAFE_TAG_RE = re.compile(r'</?[A-Za-z][A-Za-z0-9]*(?:\s[^<>]*)?/?>')
def strip_html_tags_safe(text: str) -> str:
    return html_unescape(SAFE_TAG_RE.sub('', text))

SOFT_404_RE = re.compile(r'không tìm thấy|khong tim thay|page not found|not found|页面不存在|该问题已被删除|bài viết đã bị xóa', re.I)
def is_soft_404(text: str) -> bool:
    return bool(SOFT_404_RE.search(text[:600]))

JUNK_SELECTORS = [
    'header', 'nav', 'footer',
    '.breadcrumb', '.breadcrumbs', '.breadcrumb-item',
    '.related', '.related-posts', '.related-articles', '.related-news', '.tin-lien-quan',
    '.social-share', '.share-buttons', '.social',
    '.comment', '.comments', '#comments', '.binh-luan',
    '.ads', '.adsbygoogle', '.advertisement', '.banner', '.quang-cao',
    '.booking', '.dat-lich', '.hotline', '.popup',
    '.author-info', '.byline', '.newsletter',
    'script', 'style', 'noscript', 'iframe',
    '[aria-hidden=true]', '[hidden]',
]

ROLE_MAP = {
    'toc':        re.compile(r'muc-luc|table-of-contents|toc', re.I),
    'reference':  re.compile(r'tai-lieu-tham-khao|references?|bibliography|nguon-tham-khao', re.I),
    'disclaimer': re.compile(r'mien-tru-trach-nhiem|disclaimer|khuyen-cao', re.I),
    'lead':       re.compile(r'sapo|lead|intro|mo-dau', re.I),
    'faq':        re.compile(r'faq|cau-hoi-thuong-gap', re.I),
}

def guess_role(node, text: str) -> Optional[str]:
    attrs = f"{node.attributes.get('class') or ''} {node.attributes.get('id') or ''}".lower()
    for role, pat in ROLE_MAP.items():
        if pat.search(attrs) or pat.search(text[:100]):
            return role
    return None

def safe_css(node, selector):
    try: return node.css(selector)
    except Exception: return []

def safe_css_first(node, selector):
    m = safe_css(node, selector)
    return m[0] if m else None

CJK_RE = re.compile(r'[\u4e00-\u9fff]')
CONTAINER_BLOCK_TAGS = {'p','li','h1','h2','h3','h4','h5','h6','tr','table','blockquote','div','ul','ol','section','article'}


In [ ]:
# ---- Trích xuất PubMed + HTML ----
def extract_pubmed_doc(pm_rec: Dict[str, Any], doc_id: int, url: str) -> Dict[str, Any]:
    title = normalize_ws(strip_html_tags_safe(pm_rec.get('title', '') or '').rstrip('.'))
    blocks, text_parts = [], []
    char_cursor = 0
    def add_block(btype, text, level=None, role=None):
        nonlocal char_cursor
        cs, ce = char_cursor, char_cursor + len(text)
        char_cursor = ce + 2
        blocks.append({'type': btype, 'level': level, 'role': role, 'text': text, 'char_start': cs, 'char_end': ce})
        text_parts.append(text)
    if title: add_block('heading', title, level=1)
    abs_text = pm_rec.get('abstractText') or pm_rec.get('abstract') or ''
    if isinstance(abs_text, str) and abs_text:
        if '<' in abs_text:
            parts = re.split(r'<h4[^>]*>(.*?)</h4>', abs_text)
            p0 = normalize_ws(strip_html_tags_safe(parts[0]))
            if p0: add_block('paragraph', p0)
            for lbl_raw, cnt_raw in zip(parts[1::2], parts[2::2]):
                lbl = normalize_ws(strip_html_tags_safe(lbl_raw)).rstrip(':')
                cnt = normalize_ws(strip_html_tags_safe(cnt_raw))
                if cnt: add_block('paragraph', f'{lbl}: {cnt}' if lbl else cnt)
        else:
            add_block('paragraph', normalize_ws(abs_text))
    kws = pm_rec.get('keywordList', {}).get('keyword', [])
    if kws: add_block('paragraph', 'Keywords: ' + '; '.join(str(k) for k in kws) + '.')
    full_text = '\n\n'.join(text_parts)
    flags = [] if any(b['type']=='paragraph' for b in blocks) else ['no_abstract']
    return {
        'id': doc_id, 'url': url, 'final_url': url,
        'lang': 'en', 'lang_conf': 1.0,
        'title': title, 'published_date': parse_published_date(pm_rec.get('firstPublicationDate')),
        'site': 'pubmed.ncbi.nlm.nih.gov',
        'category': ((pm_rec.get('journalInfo') or {}).get('journal') or {}).get('title'),
        'text': full_text, 'blocks': blocks, 'n_chars': len(full_text),
        'sha1_text': hashlib.sha1(full_text.encode('utf-8')).hexdigest(),
        'dup_group': None, 'extractor': 'pubmed_europepmc_v3',
        'quality_flags': flags, 'schema_version': 2,
    }


def extract_html_doc(html_bytes: bytes, encoding: str, url: str, doc_id: int) -> Dict[str, Any]:
    dom = urlparse(url).netloc.lower()
    dom_clean = re.sub(r'^www\.', '', dom).split(':')[0]
    domain_expected_lang = DOMAIN_LANG_MAP.get(dom) or DOMAIN_LANG_MAP.get(dom_clean)
    try:
        html_str = html_bytes.decode(encoding, errors='replace')
    except (LookupError, UnicodeDecodeError):
        html_str = html_bytes.decode('utf-8', errors='replace')
    tree = HTMLParser(html_str)

    # title trước khi xóa rác
    title = None
    t_hint = TITLE_HINTS.get(dom) or TITLE_HINTS.get(dom_clean)
    if t_hint:
        try:
            m = re.search(r'in\s+([^\s,]+)', t_hint)
            t_sel = m.group(1) if m else t_hint
            tn = tree.css_first(t_sel)
            if tn: title = normalize_ws(tn.text(deep=True, strip=False))
        except Exception: pass
    if not title:
        tn = tree.css_first('h1') or tree.css_first('meta[property="og:title"]') or tree.css_first('title')
        if tn:
            title = normalize_ws(tn.attributes.get('content') if tn.tag=='meta' else tn.text(deep=True, strip=False))

    # ngày
    pub_date = None
    for sel in ['meta[property="article:published_time"]','meta[name="pubdate"]','meta[name="publishdate"]','meta[itemprop="datePublished"]','meta[name="date"]']:
        m = safe_css_first(tree, sel)
        if m and m.attributes.get('content'):
            pub_date = parse_published_date(m.attributes['content'])
            if pub_date: break
    if not pub_date:
        def _find(v):
            if isinstance(v, dict):
                if v.get('datePublished'): return v['datePublished']
                for c in v.values():
                    r = _find(c)
                    if r: return r
            elif isinstance(v, list):
                for c in v:
                    r = _find(c)
                    if r: return r
            return None
        for sc in safe_css(tree, 'script[type="application/ld+json"]'):
            try:
                pub_date = parse_published_date(_find(json.loads(sc.text(deep=True, strip=True))))
                if pub_date: break
            except (ValueError, TypeError): pass
    if not pub_date:
        for sel in ['time[datetime]','time','[itemprop="datePublished"]']:
            n = safe_css_first(tree, sel)
            if n:
                pub_date = parse_published_date(n.attributes.get('datetime') or n.attributes.get('content') or n.text(deep=True, strip=True))
                if pub_date: break

    # category từ breadcrumb
    category = None
    bc = tree.css_first('.breadcrumb, .breadcrumbs, .breadcrumb-item')
    if bc: category = normalize_ws(bc.text(deep=True, strip=False))

    # xóa rác
    for sel in JUNK_SELECTORS:
        for node in tree.css(sel): node.decompose()

    # container
    content_containers = []
    hint = CONTAINER_HINTS.get(dom) or CONTAINER_HINTS.get(dom_clean)
    if hint:
        for sel in hint.get('also', []): content_containers.extend(tree.css(sel))
        if hint.get('main'): content_containers.extend(tree.css(hint['main']))
    if not content_containers:
        for sel in ['article','main','.detail-content','.article-content','.post-content','.content-detail','#content','#main-content','.main-content','.entry-content']:
            n = tree.css_first(sel)
            if n: content_containers.append(n); break
    if not content_containers:
        content_containers = [tree.css_first('body') or tree.root]

    is_qa = any(k in dom for k in ['120ask','ask.39.net'])
    blocks, text_parts = [], []
    char_cursor = 0
    def add_block(btype, btext, level=None, role=None):
        nonlocal char_cursor
        btext = normalize_ws(btext)
        if not btext: return  # giữ nguyên block có nội dung dù chỉ dài 1 ký tự
        cs, ce = char_cursor, char_cursor + len(btext)
        char_cursor = ce + 2
        blocks.append({'type': btype, 'level': level, 'role': role, 'text': btext, 'char_start': cs, 'char_end': ce})
        text_parts.append(btext)
    if title: add_block('heading', title, level=1)

    seen = set()
    for container in content_containers:
        if not container: continue
        for node in container.traverse():
            tag = getattr(node, 'tag', None)
            if not tag: continue
            if id(node) in seen: continue
            if tag == 'tr':
                seen.add(id(node))
                cells = [normalize_ws(c.text(deep=True, strip=False)) for c in node.css('th, td')]
                if any(cells): add_block('table', '\t'.join(cells), role=None)
                continue
            if node.parent and getattr(node.parent, 'tag', '') == 'tr':
                continue
            if tag in ('h1','h2','h3','h4','h5','h6'):
                seen.add(id(node))
                txt = node.text(deep=True, strip=False)
                if title and normalize_ws(txt) == title and len(blocks) == 1: continue
                add_block('heading', txt, level=int(tag[1]), role=guess_role(node, txt))
            elif tag == 'li':
                seen.add(id(node))
                depth = 0; cur = node.parent
                while cur:
                    if getattr(cur, 'tag', '') in ('ul', 'ol'): depth += 1
                    cur = cur.parent
                depth = max(depth, 1)
                txt = node.text(deep=True, strip=False)
                add_block('list_item', txt, level=depth, role=guess_role(node, txt))
            elif tag in ('p','blockquote','figcaption','dd','dt'):
                if node.parent and getattr(node.parent, 'tag', '') in ('li','tr'): continue
                seen.add(id(node))
                txt = node.text(deep=True, strip=False)
                if not txt: continue
                role = guess_role(node, txt)
                if is_qa:
                    cls_id = f"{node.attributes.get('class') or ''} {node.attributes.get('id') or ''}".lower()
                    if 'ask' in cls_id or 'question' in cls_id: add_block('qa_q', txt, role=role)
                    elif 'answer' in cls_id or 'reply' in cls_id or 'doctor' in cls_id: add_block('qa_a', txt, role=role)
                    else: add_block('paragraph', txt, role=role)
                else:
                    add_block('paragraph', txt, role=role)
            elif tag == 'div':
                has_child_block = any(getattr(c,'tag','') in CONTAINER_BLOCK_TAGS for c in node.iter() if c is not node)
                if not has_child_block:
                    txt = node.text(deep=True, strip=False)
                    if txt and txt.strip():
                        seen.add(id(node))
                        add_block('paragraph', txt, role=guess_role(node, txt))

    full_text = '\n\n'.join(text_parts)
    flags = []
    if len(full_text) < 200: flags.append('short')
    if is_soft_404(full_text): flags.append('soft_404')

    cjk_count = len(CJK_RE.findall(full_text[:1000]))
    if cjk_count > 10:
        lang, lang_conf = 'zh', 0.99
    elif domain_expected_lang:
        lang, lang_conf = domain_expected_lang, 0.95
    elif any(dom.endswith(s) for s in ('.cn','.com.cn')):
        lang, lang_conf = 'zh', 0.95
    elif 'pubmed' in dom:
        lang, lang_conf = 'en', 1.0
    else:
        lang, lang_conf = 'vi', 0.95

    return {
        'id': doc_id, 'url': url, 'final_url': url,
        'lang': lang, 'lang_conf': lang_conf,
        'title': title, 'published_date': pub_date,
        'site': dom, 'category': category,
        'text': full_text, 'blocks': blocks, 'n_chars': len(full_text),
        'sha1_text': hashlib.sha1(full_text.encode('utf-8')).hexdigest(),
        'dup_group': None, 'extractor': 'custom_cleaner_v3',
        'quality_flags': flags, 'schema_version': 2,
    }


In [ ]:
# ---- Gold set: trích xuất gold_html.zip trước khi chạy clean toàn corpus ----
# Chạy sau cells 24, 26, 27. Không mở/ghi crawl_status và không khởi chạy crawler.
from zipfile import ZipFile
from collections import defaultdict
import subprocess
import sys
import time
import xml.etree.ElementTree as ET

_gold_candidates = [
    Path("gold_html.zip"),
    Path.home() / "Downloads" / "gold_html.zip",
    Path(DATA_ROOT) / "gold_html.zip",
]
GOLD_HTML_ZIP = next((p for p in _gold_candidates if p.is_file()), None)
if GOLD_HTML_ZIP is None:
    raise FileNotFoundError(
        "Không tìm thấy gold_html.zip. Đặt file ở thư mục notebook, Downloads, "
        "hoặc DATA_ROOT."
    )

try:
    import trafilatura as _trafilatura
except ImportError:
    _trafilatura = None
    logger.warning("Thiếu trafilatura; bỏ qua extractor baseline này.")
try:
    from resiliparse.extract.html2text import extract_plain_text as _resiliparse_text
except ImportError:
    _resiliparse_text = None
    logger.warning("Thiếu resiliparse; bỏ qua extractor baseline này.")

GOLD_OUTPUT_ROOT = REPORT_DIR / "extract_gold"
GOLD_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
_gold_engine_dirs = {"custom_cleaner_v3": GOLD_OUTPUT_ROOT / "custom_cleaner_v3"}
if _trafilatura is not None:
    _gold_engine_dirs["trafilatura"] = GOLD_OUTPUT_ROOT / "trafilatura"
if _resiliparse_text is not None:
    _gold_engine_dirs["resiliparse"] = GOLD_OUTPUT_ROOT / "resiliparse"
for _dir in _gold_engine_dirs.values():
    _dir.mkdir(parents=True, exist_ok=True)

def _gold_write_doc(engine: str, doc: Dict[str, Any]) -> None:
    out_path = _gold_engine_dirs[engine] / f"{int(doc['id'])}.json"
    out_path.write_text(
        json.dumps(doc, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )

def _gold_make_doc(doc_id: int, url: str, engine: str, items: List[Tuple[str, str]]) -> Dict[str, Any]:
    blocks, parts, cursor = [], [], 0
    for block_type, raw_text in items:
        text = normalize_ws(str(raw_text or ""))
        if not text:
            continue
        start, end = cursor, cursor + len(text)
        blocks.append({
            "type": block_type, "level": None, "role": None,
            "text": text, "char_start": start, "char_end": end,
        })
        parts.append(text)
        cursor = end + 2
    full_text = "\n\n".join(parts)
    return {
        "id": int(doc_id), "url": url, "final_url": url,
        "lang": "unknown", "lang_conf": 0.0,
        "title": None, "published_date": None,
        "site": urlparse(url).netloc.lower(), "category": None,
        "text": full_text, "blocks": blocks, "n_chars": len(full_text),
        "sha1_text": hashlib.sha1(full_text.encode("utf-8")).hexdigest(),
        "dup_group": None, "extractor": engine,
        "quality_flags": ["gold_baseline"], "schema_version": 2,
    }

_gold_timings = defaultdict(list)
_gold_errors = defaultdict(int)
with ZipFile(GOLD_HTML_ZIP, "r") as _gold_zip:
    if "manifest.csv" not in _gold_zip.namelist():
        raise ValueError(f"{GOLD_HTML_ZIP} thiếu manifest.csv")
    _gold_manifest = pd.read_csv(_gold_zip.open("manifest.csv"), dtype={"id": "int64"})
    _required_cols = {"id", "lang", "domain", "url"}
    if not _required_cols.issubset(_gold_manifest.columns):
        raise ValueError(f"manifest.csv cần các cột: {sorted(_required_cols)}")

    _html_rows = _gold_manifest[
        ~_gold_manifest["domain"].astype(str).str.lower().eq("pubmed.ncbi.nlm.nih.gov")
    ]
    for _row in tqdm(_html_rows.itertuples(index=False), total=len(_html_rows), desc="Gold HTML"):
        _doc_id, _url = int(_row.id), str(_row.url)
        _html_name = f"html/{_doc_id}.html"
        _meta_name = f"html/{_doc_id}.meta.json"
        if _html_name not in _gold_zip.namelist():
            logger.warning(f"Gold thiếu {_html_name}; bỏ qua ID {_doc_id}")
            continue
        _html_bytes = _gold_zip.read(_html_name)
        _meta = json.loads(_gold_zip.read(_meta_name).decode("utf-8")) if _meta_name in _gold_zip.namelist() else {}
        _encoding = detect_encoding(_html_bytes, _meta.get("content_type", ""))
        _html_text = _html_bytes.decode(_encoding, errors="replace")

        _t0 = time.perf_counter()
        try:
            _doc = extract_html_doc(_html_bytes, _encoding, _url, _doc_id)
            _gold_write_doc("custom_cleaner_v3", _doc)
        except Exception:
            _gold_errors["custom_cleaner_v3"] += 1
            logger.exception(f"custom_cleaner_v3 lỗi ID {_doc_id}")
        finally:
            _gold_timings["custom_cleaner_v3"].append(time.perf_counter() - _t0)

        if _trafilatura is not None:
            _t0 = time.perf_counter()
            try:
                _xml = _trafilatura.extract(
                    _html_text, url=_url, output_format="xml",
                    include_comments=False, include_tables=True,
                    include_formatting=True, favor_recall=True,
                )
                _items = []
                if _xml:
                    _root = ET.fromstring(_xml)
                    _main = _root.find(".//main")
                    _main = _main if _main is not None else _root
                    _block_tags = {"head", "p", "quote", "item", "cell"}
                    for _node in _main.iter():
                        _tag = _node.tag.rsplit("}", 1)[-1] if isinstance(_node.tag, str) else ""
                        if _tag not in _block_tags:
                            continue
                        _desc = list(_node.iter())[1:]
                        if any((d.tag.rsplit("}", 1)[-1] if isinstance(d.tag, str) else "") in _block_tags for d in _desc):
                            continue
                        _text = "".join(_node.itertext())
                        _items.append(("heading" if _tag == "head" else "paragraph", _text))
                _gold_write_doc("trafilatura", _gold_make_doc(_doc_id, _url, "trafilatura", _items))
            except Exception:
                _gold_errors["trafilatura"] += 1
                logger.exception(f"trafilatura lỗi ID {_doc_id}")
            finally:
                _gold_timings["trafilatura"].append(time.perf_counter() - _t0)

        if _resiliparse_text is not None:
            _t0 = time.perf_counter()
            try:
                _plain = _resiliparse_text(
                    _html_text, preserve_formatting=True, main_content=True,
                    list_bullets=False, alt_texts=False, links=False,
                ) or ""
                _items = [
                    ("paragraph", _part)
                    for _part in re.split(r"\n\s*\n+", _plain.strip())
                    if _part.strip()
                ]
                _gold_write_doc("resiliparse", _gold_make_doc(_doc_id, _url, "resiliparse", _items))
            except Exception:
                _gold_errors["resiliparse"] += 1
                logger.exception(f"resiliparse lỗi ID {_doc_id}")
            finally:
                _gold_timings["resiliparse"].append(time.perf_counter() - _t0)

# 8 PMID trong Gold phải đi qua Europe PMC API, không dùng bộ trích HTML.
_gold_pm_rows = _gold_manifest[
    _gold_manifest["domain"].astype(str).str.lower().eq("pubmed.ncbi.nlm.nih.gov")
]
if len(_gold_pm_rows):
    if "fetch_pubmed_batches" not in globals():
        logger.warning("Chưa có fetch_pubmed_batches; chạy cell 24 rồi chạy lại cell Gold.")
    else:
        _gold_pmids = [
            str(urlparse(str(u)).path.rstrip("/").split("/")[-1])
            for u in _gold_pm_rows["url"]
        ]
        _t0 = time.perf_counter()
        try:
            _gold_hits = fetch_pubmed_batches(_gold_pmids, batch_size=max(1, len(_gold_pmids)), max_batches=1)
            _gold_by_pmid = {str(h.get("pmid", "")).strip(): h for h in _gold_hits}
            for _row in _gold_pm_rows.itertuples(index=False):
                _pmid = str(urlparse(str(_row.url)).path.rstrip("/").split("/")[-1])
                _hit = _gold_by_pmid.get(_pmid)
                if _hit is None:
                    logger.warning(f"Europe PMC không trả PMID {_pmid}")
                    continue
                _doc = extract_pubmed_doc(_hit, int(_row.id), str(_row.url))
                for _engine in _gold_engine_dirs:
                    _gold_write_doc(_engine, _doc)
            _gold_timings["europepmc_api"].append(time.perf_counter() - _t0)
        except Exception:
            logger.exception("Không xử lý được PubMed subset trong Gold.")

print(f"Gold archive: {GOLD_HTML_ZIP}")
print(f"HTML pages: {len(_html_rows):,}; PubMed pages: {len(_gold_pm_rows):,}")
print(f"Prediction files: {GOLD_OUTPUT_ROOT}")
for _engine in _gold_engine_dirs:
    _times = _gold_timings.get(_engine, [])
    _avg = (sum(_times) / len(_times)) if _times else 0.0
    print(f"  {_engine}: avg={_avg:.3f}s/page | errors={_gold_errors[_engine]}")

_gold_answers = Path(DATA_ROOT) / "gold"
if (_gold_answers / "manifest.csv").is_file() and any(_gold_answers.glob("*.txt")):
    _score_out = REPORT_DIR / "extract_scores"
    subprocess.run(
        [sys.executable, str(Path("eval/score_extraction.py")),
         "--gold", str(_gold_answers), "--pred", str(GOLD_OUTPUT_ROOT),
         "--out", str(_score_out)],
        check=False,
    )
else:
    print("Chưa chấm điểm: gold_html.zip chỉ chứa HTML; cần thư mục gold/ có manifest.csv và <id>.txt.")

In [ ]:
# ---- Bóc tách snapshot từ shard đã đóng và/hoặc nguồn API ----
# Khi full crawl đang chạy ở kernel khác, đặt DOCS_SOURCE_MODE='html_raw' và
# DOCS_MAX_RECORDS=10_000 để trích mẫu từ các shard đã được atomic-write.
# Khi xử lý cuối: DOCS_SOURCE_MODE='all', DOCS_MAX_RECORDS=None,
# DOCS_COVERAGE_MODE='all_source'.
DOCS_SOURCE_MODE = 'all'          # 'all' | 'html_raw'
DOCS_MAX_RECORDS: Optional[int] = None
DOCS_COVERAGE_MODE = 'all_source' # 'all_source' | 'available_raw'
if DOCS_SOURCE_MODE not in {'all', 'html_raw'}:
    raise ValueError('DOCS_SOURCE_MODE phải là all hoặc html_raw')
if DOCS_COVERAGE_MODE not in {'all_source', 'available_raw'}:
    raise ValueError('DOCS_COVERAGE_MODE phải là all_source hoặc available_raw')
if DOCS_MAX_RECORDS is not None and DOCS_MAX_RECORDS <= 0:
    raise ValueError('DOCS_MAX_RECORDS phải dương hoặc None')
if DOCS_COVERAGE_MODE == 'all_source' and (DOCS_SOURCE_MODE != 'all' or DOCS_MAX_RECORDS is not None):
    raise ValueError('Trích mẫu/HTML-only phải dùng DOCS_COVERAGE_MODE=available_raw')

id_to_url = dict(zip(df_links['id'], df_links['url']))
processed_docs: List[Dict[str, Any]] = []

# PubMed chỉ được trích khi chọn toàn bộ nguồn.
if DOCS_SOURCE_MODE == 'all':
    pmid_to_row = {str(r['pmid']): r for _, r in pubmed_links.iterrows()}
    for rec in pubmed_records:
        if DOCS_MAX_RECORDS is not None and len(processed_docs) >= DOCS_MAX_RECORDS:
            break
        pmid = str(rec.get('pmid',''))
        if pmid in pmid_to_row:
            row = pmid_to_row[pmid]
            processed_docs.append(extract_pubmed_doc(rec, int(row['id']), row['url']))

# Merge dữ liệu chính thức zysjonline theo PHAN_CONG.md.
ZYS_PATH = DOCS_DIR / 'docs_zysjonline.parquet'
if DOCS_SOURCE_MODE == 'all' and ZYS_PATH.exists():
    zys_df = pd.read_parquet(ZYS_PATH)
    if 'id' not in zys_df.columns or 'text' not in zys_df.columns:
        raise ValueError(f'{ZYS_PATH} cần có ít nhất cột id và text')
    zys_df['id'] = pd.to_numeric(zys_df['id'], errors='raise').astype('int64')
    zys_df['text'] = zys_df['text'].fillna('').astype(str)
    zys_df['url'] = zys_df['id'].map(id_to_url).fillna(zys_df['url'] if 'url' in zys_df else '')
    defaults = {
        'final_url': None, 'lang': 'zh', 'lang_conf': 1.0, 'title': None,
        'published_date': None, 'site': 'zysjonline.com', 'category': None,
        'dup_group': None, 'extractor': 'zysjonline_official', 'schema_version': 2,
    }
    for col, default in defaults.items():
        if col not in zys_df.columns:
            zys_df[col] = default
    zys_df['final_url'] = zys_df['final_url'].fillna(zys_df['url'])
    zys_df['site'] = zys_df['site'].fillna('zysjonline.com')
    zys_df['extractor'] = 'zysjonline_official'
    zys_df['schema_version'] = 2
    if 'blocks' not in zys_df.columns:
        zys_df['blocks'] = [[] for _ in range(len(zys_df))]
    def _zys_blocks(value, text):
        if isinstance(value, (list, tuple)):
            return list(value)
        if not text:
            return []
        return [{'type': 'paragraph', 'level': None, 'role': None, 'text': text,
                 'char_start': 0, 'char_end': len(text)}]
    zys_df['blocks'] = [_zys_blocks(b, t) for b, t in zip(zys_df['blocks'], zys_df['text'])]
    if 'n_chars' not in zys_df.columns:
        zys_df['n_chars'] = zys_df['text'].str.len()
    else:
        zys_df['n_chars'] = pd.to_numeric(zys_df['n_chars'], errors='coerce').fillna(zys_df['text'].str.len())
    if 'sha1_text' not in zys_df.columns:
        zys_df['sha1_text'] = zys_df['text'].apply(lambda t: hashlib.sha1(t.encode('utf-8')).hexdigest() if t else None)
    else:
        zys_df['sha1_text'] = [h if isinstance(h, str) and h else (hashlib.sha1(t.encode('utf-8')).hexdigest() if t else None)
                               for h, t in zip(zys_df['sha1_text'], zys_df['text'])]
    if 'quality_flags' not in zys_df.columns:
        zys_df['quality_flags'] = [[] for _ in range(len(zys_df))]
    else:
        zys_df['quality_flags'] = [v if isinstance(v, (list, tuple)) or (hasattr(v, 'tolist') and not isinstance(v, str)) else [] for v in zys_df['quality_flags']]
        zys_df['quality_flags'] = [v.tolist() if hasattr(v, 'tolist') else list(v) for v in zys_df['quality_flags']]
    for col in ['lang', 'title', 'published_date', 'category']:
        if col in zys_df.columns:
            zys_df[col] = zys_df[col].where(zys_df[col].notna(), defaults.get(col))
    zys_records = zys_df.to_dict('records')
    if DOCS_MAX_RECORDS is not None:
        zys_records = zys_records[:max(0, DOCS_MAX_RECORDS - len(processed_docs))]
    processed_docs.extend(zys_records)
    logger.info(f'Đã gộp {len(zys_records):,} bản ghi zysjonline từ {ZYS_PATH}')
elif DOCS_SOURCE_MODE == 'all':
    logger.warning(f'Không tìm thấy {ZYS_PATH}; ID zysjonline sẽ được đánh dấu placeholder ở bước coverage')

# HTML shards
shard_files = sorted(HTML_DIR.glob('shard_*.jsonl.zst'))
logger.info(f'Xử lý {len(shard_files)} shard HTML')
dctx = zstd.ZstdDecompressor()
for sf in tqdm(shard_files, desc='Clean closed raw shards'):
    if DOCS_MAX_RECORDS is not None and len(processed_docs) >= DOCS_MAX_RECORDS:
        break
    try:
        with open(sf, 'rb') as f, dctx.stream_reader(f) as reader:
            raw = reader.read().decode('utf-8')
    except Exception as e:
        logger.exception(f'Lỗi đọc {sf}: {e}')
        continue
    for line in raw.splitlines():
        if DOCS_MAX_RECORDS is not None and len(processed_docs) >= DOCS_MAX_RECORDS:
            break
        if not line.strip(): continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        doc_id = rec['id']
        url = id_to_url.get(doc_id, rec.get('final_url',''))
        content_type = (rec.get('content_type') or '').lower()
        if 'text/html' not in content_type and 'application/xhtml' not in content_type:
            processed_docs.append({
                'id': doc_id, 'url': url, 'final_url': rec.get('final_url', url),
                'lang': DOMAIN_LANG_MAP.get(urlparse(url).netloc.lower(), 'unknown'), 'lang_conf': 0.0,
                'title': None, 'published_date': None, 'site': urlparse(url).netloc.lower(),
                'category': None, 'text': '', 'blocks': [], 'n_chars': 0,
                'sha1_text': None, 'dup_group': None, 'extractor': 'non_html',
                'quality_flags': ['non_html'], 'schema_version': 2,
            })
            continue
        try:
            raw_b = base64.b64decode(rec['body_b64'])
            doc = extract_html_doc(raw_b, rec.get('encoding','utf-8'), url, doc_id)
            processed_docs.append(doc)
        except Exception as e:
            dom = urlparse(url).netloc.lower()
            processed_docs.append({
                'id': doc_id, 'url': url, 'final_url': url,
                'lang': DOMAIN_LANG_MAP.get(dom, 'vi'), 'lang_conf': 0.0,
                'title': None, 'published_date': None,
                'site': dom, 'category': None,
                'text': '', 'blocks': [], 'n_chars': 0,
                'sha1_text': hashlib.sha1(b'').hexdigest(),
                'dup_group': None, 'extractor': 'custom_cleaner_v3',
                'quality_flags': ['extract_error'], 'schema_version': 2,
            })

logger.info(f'Tổng documents đã trích xuất: {len(processed_docs):,}')
print(f'Documents: {len(processed_docs):,}')


---
## Bước 4 — Dedup & xuất Parquet (T1.3)

- sha1_text → `dup_group` (không xóa).
- Bất biến `text[char_start:char_end] == block.text`.
- PyArrow schema v2: `blocks` = `list<struct>`, `quality_flags` = `list<string>`.


In [ ]:
# ---- Coverage ID nguồn, placeholder có cờ, và dedup không xoá ----
df_docs = pd.DataFrame(processed_docs)
logger.info(f'Docs trước coverage: {len(df_docs):,}')

source_meta = df_links[['id', 'url']].copy()
source_meta['id'] = pd.to_numeric(source_meta['id'], errors='raise').astype('int64')
if source_meta['id'].duplicated().any():
    repeated_source_ids = source_meta.loc[source_meta['id'].duplicated(keep=False), 'id'].astype('int64').unique().tolist()
    raise AssertionError(f'df_links có ID trùng: {repeated_source_ids[:20]}')
all_source_ids = set(source_meta['id'].tolist())

if len(df_docs) > 0:
    if 'id' not in df_docs.columns:
        raise ValueError('Documents thiếu cột id')
    df_docs['id'] = pd.to_numeric(df_docs['id'], errors='raise').astype('int64')
    df_docs['_source_order'] = range(len(df_docs))
    df_docs['_text_len'] = df_docs.get('n_chars', pd.Series(0, index=df_docs.index)).fillna(0)
    extractor = df_docs.get('extractor', pd.Series('', index=df_docs.index)).fillna('').astype(str)
    df_docs['_source_priority'] = 10
    df_docs.loc[extractor.eq('placeholder'), '_source_priority'] = 0
    df_docs.loc[extractor.eq('custom_cleaner_v3') & (df_docs['_text_len'] > 0), '_source_priority'] = 40
    df_docs.loc[extractor.str.startswith('pubmed_europepmc'), '_source_priority'] = 50
    df_docs.loc[extractor.eq('zysjonline_official'), '_source_priority'] = 50
    duplicate_mask = df_docs['id'].duplicated(keep=False)
    if duplicate_mask.any():
        duplicate_audit = df_docs.loc[duplicate_mask, ['id', 'extractor', 'n_chars', 'url']].copy()
        duplicate_audit.to_csv(REPORT_DIR / 'duplicate_document_ids.csv', index=False)
        logger.warning(f'Có {duplicate_audit["id"].nunique():,} ID có nhiều bản trích; lưu audit và chọn nguồn ưu tiên/lần xuất hiện sau')
        df_docs = (df_docs.sort_values(['id', '_source_priority', '_source_order'], kind='stable')
                          .drop_duplicates('id', keep='last'))
    df_docs = df_docs.drop(columns=['_source_order', '_text_len', '_source_priority'])
else:
    df_docs = pd.DataFrame(columns=['id', 'url', 'text', 'blocks', 'n_chars', 'extractor'])

have_ids = set(df_docs['id'].astype('int64').tolist()) if len(df_docs) else set()
expected_ids = all_source_ids if DOCS_COVERAGE_MODE == 'all_source' else have_ids
missing_ids = expected_ids - have_ids
extra_ids = have_ids - all_source_ids
if extra_ids:
    raise AssertionError(f'Documents có {len(extra_ids):,} ID không thuộc df_links: {list(extra_ids)[:20]}')

if missing_ids:
    logger.warning(f'Thiếu {len(missing_ids):,} ID nguồn; tạo placeholder để giữ coverage và đánh dấu rõ là chưa có nội dung')
    missing_meta = source_meta[source_meta['id'].isin(missing_ids)].copy()
    con.register('_missing_source_ids', missing_meta[['id']])
    try:
        status_meta = con.execute("""
            SELECT m.id, cs.status
            FROM _missing_source_ids m
            LEFT JOIN crawl_status cs USING (id)
        """).df()
    finally:
        con.unregister('_missing_source_ids')
    missing_meta = missing_meta.merge(status_meta, on='id', how='left')
    placeholders = []
    for r in missing_meta.itertuples(index=False):
        url = str(r.url or '')
        dom = urlparse(url).netloc.lower().removeprefix('www.') if url else ''
        status = r.status if isinstance(r.status, str) and r.status else 'not_in_status'
        placeholders.append({
            'id': int(r.id), 'url': url, 'final_url': url,
            'lang': DOMAIN_LANG_MAP.get(dom, 'unknown'), 'lang_conf': 0.0,
            'title': None, 'published_date': None, 'site': dom, 'category': None,
            'text': '', 'blocks': [], 'n_chars': 0, 'sha1_text': None,
            'dup_group': None, 'extractor': 'placeholder',
            'quality_flags': ['missing', f'final_status:{status}'], 'schema_version': 2,
        })
    df_docs = pd.concat([df_docs, pd.DataFrame(placeholders)], ignore_index=True)

# Bất biến đầu ra: đúng tập ID nguồn, mỗi ID đúng một hàng.
final_ids = pd.to_numeric(df_docs['id'], errors='raise').astype('int64')
missing_final = expected_ids - set(final_ids.tolist())
extra_final = set(final_ids.tolist()) - all_source_ids
dup_final = final_ids[final_ids.duplicated(keep=False)].unique().tolist()
assert not missing_final, f'Còn thiếu {len(missing_final):,} ID trong scope: {list(missing_final)[:10]}'
assert not extra_final, f'Có {len(extra_final):,} ID lạ: {list(extra_final)[:10]}'
assert not dup_final, f'Còn {len(dup_final):,} ID trùng sau xử lý: {dup_final[:10]}'
logger.info(f'✅ ID coverage ({DOCS_COVERAGE_MODE}): {len(df_docs):,}/{len(expected_ids):,}; '
            f'placeholders={(df_docs["extractor"] == "placeholder").sum():,}')

# Exact-text duplicate group; placeholders/nội dung rỗng không bị gom thành một nhóm trùng.
df_docs['dup_group'] = None
nonempty = df_docs['text'].fillna('').astype(str).str.len().gt(0)
if 'extractor' in df_docs.columns:
    nonempty &= df_docs['extractor'].fillna('').astype(str).ne('placeholder')
sha1_map: Dict[str, int] = {}
for _, r in df_docs.loc[nonempty].iterrows():
    digest = r.get('sha1_text')
    if isinstance(digest, str) and digest and digest not in sha1_map:
        sha1_map[digest] = int(r['id'])
df_docs['dup_group'] = df_docs['sha1_text'].map(lambda s: sha1_map.get(s) if isinstance(s, str) and s else None)
df_docs.loc[df_docs['dup_group'] == df_docs['id'], 'dup_group'] = None
df_docs.loc[~nonempty, 'dup_group'] = None
logger.info(f'Dedup sha1 (chỉ gắn nhóm, không xóa): {int(df_docs["dup_group"].notna().sum()):,} / {len(df_docs):,}')


In [ ]:
# ---- Kiểm tra bất biến toạ độ block trên TOÀN BỘ documents ----
def check_invariant_full(row) -> List[str]:
    errors = []
    raw_text = row.get('text', '')
    text = raw_text if isinstance(raw_text, str) else ''
    raw_blocks = row.get('blocks', [])
    blocks = list(raw_blocks) if isinstance(raw_blocks, (list, tuple)) else []

    try:
        n_chars = int(row.get('n_chars'))
    except (TypeError, ValueError):
        n_chars = -1
    if n_chars != len(text):
        errors.append(f'n_chars={n_chars} != len(text)={len(text)}')

    previous_end = None
    for i, block in enumerate(blocks):
        if not isinstance(block, dict):
            errors.append(f'block[{i}] không phải dict')
            continue
        try:
            start = int(block['char_start'])
            end = int(block['char_end'])
            value = block['text']
        except (KeyError, TypeError, ValueError):
            errors.append(f'block[{i}] thiếu/sai trường offset hoặc text')
            continue
        if not isinstance(value, str):
            errors.append(f'block[{i}].text không phải chuỗi')
            continue
        if not (0 <= start <= end <= len(text)):
            errors.append(f'block[{i}] biên ngoài text: {start},{end}, len={len(text)}')
            continue
        if end - start != len(value):
            errors.append(f'block[{i}] độ dài slice không khớp text')
        if text[start:end] != value:
            errors.append(f'block[{i}] slice mismatch')
        if i == 0 and start != 0:
            errors.append(f'block đầu bắt đầu tại {start}, cần 0')
        if previous_end is not None and start != previous_end + 2:
            errors.append(f'block[{i}] gap/overlap sai: start={start}, expected={previous_end + 2}')
        previous_end = end

    block_text = '\n\n'.join(b['text'] for b in blocks if isinstance(b, dict) and isinstance(b.get('text'), str))
    if block_text != text:
        errors.append('coverage: join(block.text) != text')
    return errors

if len(df_docs) > 0:
    BATCH = 50_000
    all_errors = []
    for start in tqdm(range(0, len(df_docs), BATCH), desc='Invariant toàn bộ'):
        sub = df_docs.iloc[start:start + BATCH]
        for _, row in sub.iterrows():
            errs = check_invariant_full(row)
            if errs:
                all_errors.append({'id': int(row['id']), 'errors': '; '.join(errs)})
    if all_errors:
        err_df = pd.DataFrame(all_errors)
        err_df.to_csv(REPORT_DIR / 'invariant_violations.csv', index=False)
        raise AssertionError(f'{len(all_errors):,} documents vi phạm invariant; xem {REPORT_DIR / "invariant_violations.csv"}')
    print(f'✅ Invariant 100% cho {len(df_docs):,} documents')
else:
    print('Không có documents để kiểm tra invariant.')


In [ ]:
# ---- Xuất Parquet schema v2 vào thư mục run riêng, không ghi đè ----
import pyarrow as pa
import pyarrow.parquet as pq
import uuid

if len(df_docs) > 0:
    block_type = pa.struct([
        ('type', pa.string()),
        ('level', pa.int64()),
        ('role', pa.string()),
        ('text', pa.string()),
        ('char_start', pa.int64()),
        ('char_end', pa.int64()),
    ])
    pa_schema = pa.schema([
        ('id', pa.int64()),
        ('url', pa.string()),
        ('final_url', pa.string()),
        ('lang', pa.string()),
        ('lang_conf', pa.float64()),
        ('title', pa.string()),
        ('published_date', pa.string()),
        ('site', pa.string()),
        ('category', pa.string()),
        ('text', pa.string()),
        ('blocks', pa.list_(block_type)),
        ('n_chars', pa.int64()),
        ('sha1_text', pa.string()),
        ('dup_group', pa.int64()),
        ('extractor', pa.string()),
        ('quality_flags', pa.list_(pa.string())),
        ('schema_version', pa.int64()),
    ])

    RUN_ID = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
    RUN_DIR = DOCS_DIR / f'run_{RUN_ID}'
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    DOCS_SHARD_SIZE = 50_000
    written = 0
    shard_names = []
    n_shards = (len(df_docs) + DOCS_SHARD_SIZE - 1) // DOCS_SHARD_SIZE
    for i, start in enumerate(range(0, len(df_docs), DOCS_SHARD_SIZE)):
        sub = df_docs.iloc[start:start + DOCS_SHARD_SIZE].copy()
        tbl = pa.Table.from_pandas(sub, schema=pa_schema, preserve_index=False)
        out = RUN_DIR / f'docs_{i:05d}.parquet'
        tmp = out.with_suffix(out.suffix + '.tmp')
        pq.write_table(tbl, tmp, compression='zstd')
        os.replace(tmp, out)
        shard_names.append(out.name)
        logger.info(f'Ghi {out.name} ({len(sub):,} docs, {out.stat().st_size / 1e6:.2f} MB)')
        written += len(sub)

    extractor_counts = {str(k): int(v) for k, v in df_docs['extractor'].value_counts(dropna=False).items()}
    manifest = {
        'run_id': RUN_ID,
        'run_ts': datetime.now().isoformat(timespec='seconds'),
        'run_dir': RUN_DIR.name,
        'n_docs': int(len(df_docs)),
        'n_shards': int(n_shards),
        'shards': shard_names,
        'schema_version': 2,
        'n_placeholders': int((df_docs['extractor'] == 'placeholder').sum()),
        'extractors': extractor_counts,
    }
    manifest_path = RUN_DIR / 'manifest.json'
    manifest_tmp = manifest_path.with_suffix('.json.tmp')
    manifest_tmp.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(manifest_tmp, manifest_path)
    latest_tmp = DOCS_DIR / 'latest_run.json.tmp'
    latest_tmp.write_text(json.dumps({'run_id': RUN_ID, 'run_dir': RUN_DIR.name}, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(latest_tmp, DOCS_DIR / 'latest_run.json')
    print(f'Đã xuất {written:,} documents -> {RUN_DIR}')
    print(f'Run ID: {RUN_ID}')
else:
    print('Không có documents để xuất Parquet.')


In [ ]:
# ---- Quality dashboard ----
if len(df_docs) > 0:
    lines = [
        '# Quality Dashboard',
        f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
        '',
        '## 1. Tổng quan',
        f'- Tổng documents: {len(df_docs):,}',
        f'- Có nội dung: {(df_docs["n_chars"] > 0).sum():,}',
        f'- Median n_chars: {df_docs["n_chars"].median():.0f}',
        f'- Mean n_chars: {df_docs["n_chars"].mean():.0f}',
        f'- Bản ghi trùng (dup_group != null): {df_docs["dup_group"].notna().sum():,}',
        '',
        '## 2. Extractor',
        df_docs['extractor'].value_counts().reset_index().to_markdown(index=False),
        '',
        '## 3. Ngôn ngữ',
        df_docs['lang'].value_counts().reset_index().to_markdown(index=False),
        '',
        '## 4. Quality flags (top)',
        pd.Series([f for flags in df_docs['quality_flags'] for f in (flags or [])]).value_counts().head(20).reset_index().to_markdown(index=False),
    ]
    (REPORT_DIR / 'quality_dashboard.md').write_text('\n'.join(lines), encoding='utf-8')
    logger.info(f'Đã ghi {REPORT_DIR / "quality_dashboard.md"}')
    display(df_docs[['id','site','lang','n_chars','extractor']].head(10))


---
## Vận hành theo review 20–21/10

### 1. PubMed — chạy trước

Sau setup và tải corpus, chạy cell **PubMed qua Europe PMC REST API** ngay trước phần crawler. Mặc định `RUN_PUBMED=True`, `PUBMED_MAX_BATCHES=None`: dùng lại tất cả cache PMID cũ/mới trong `crawl/pubmed/`, chỉ tải phần thiếu, tối đa 1.000 PMID/request. Batch được ghi atomic để resume. Xem `pubmed_coverage.csv` và `pubmed_missing_pmids.csv`; số batch không thay cho số PMID thực nhận.

### 2. Crawl HTML — sẵn sàng chạy

- Chạy setup DuckDB, domain plan, seed/ngôn ngữ, encoding, limiter, fetch, shard writer và **Orchestrator: run_crawler_pipeline**. Colab giữ hàm cũ trong RAM: cần chạy lại cell Orchestrator rồi mới chạy cell chọn URL và cell chạy crawler.
- `RUN_CRAWL=True` mặc định: thực thi cell chạy crawler sẽ bắt đầu thu thập HTML. `CRAWL_MODE='full'`, `FULL_CRAWL_APPROVED=True`; full dùng toàn bộ URL còn đủ điều kiện, gồm ID 39.net ngoài mẫu 40K, không hard-code số khoảng 4.750.
- Mọi subdomain 39.net dùng chung 1 request/giây và một limiter. `lang_expected='zh'` được sửa cả cho ID đã seed. Gặp `verify.html`, 403 hoặc 429 thì dừng domain; không tự đặt `blocked` về `pending`.
- `nhathuoclongchau.com.vn`, `tamanhhospital.vn` và subdomain nằm ngoài crawler/probe thường. Không có bước reset Long Châu. Bỏ Pharmacity và zysjonline; zysjonline dùng dump chính thức. 120ask giữ `ALLOW_CLOUD_DOMAIN=False`, đồng bộ shard và status từ VPS trước khi ghép dữ liệu.
- `suckhoedoisong.vn`, `thanhnien.vn` dùng `http2=False`, tiếp tục lỗi kết nối tạm thời còn ngân sách. iiyi probe riêng tối đa một lần/domain/24 giờ.
- Checkpoint ghi shard → status → DuckDB `CHECKPOINT` → metrics. Worker có `nonlocal last_save`. URL chưa gửi request do domain halt giữ nguyên trạng thái. File `crawl/STOP` hoặc Ctrl-C yêu cầu dừng mềm.
- Sau mỗi lượt chạy, xem **Trạng thái toàn corpus theo domain**. Hết queue không có nghĩa đủ 1,2 triệu URL; PubMed có báo cáo coverage riêng.

### 3. Long Châu ở nhà — chạy song song khi nhận file

Chỉ chạy trên máy/mạng gia đình. `longchau_ranked_tv.csv` đã chia theo **vị trí lẻ trong danh sách xếp hạng**, không lọc lại bằng `id % 2`. Thử trang chủ một lần; chỉ chạy khi HTTP 200. HTTP 403 thì dừng, không thử liên tục.

Giữ nguyên lệnh theo mục 4 của review:

```bash
python crawl/crawl_home.py --corpus longchau_ranked_tv.csv --out crawl_home_tv --rate nhathuoclongchau.com.vn=0.3 --ramp 0.05 0.025 20 --workers 1 --pauses --shard-prefix tv
```

Bắt đầu 0,05 request/giây, mỗi 20 phút tăng 0,025 tới tối đa 0,3; bị chặn lần đầu tự dừng. Đồng bộ `crawl_home_tv/` vào Drive `crawl/html_home_tv/`; phần NT vào `crawl/html_home_nt/`. Cell **Hợp nhất shard crawl ở nhà** nhận cả hai nguồn, kiểm body/SHA-1, tránh nhập trùng và giữ status mới hơn.

### 4. Trích full, gold và nộp dữ liệu

Dùng **03_gold_review_and_sample2.ipynb**, build `2026-10-20-v26-shard-html-selection-postfilter-v1.12`, để chạy full đã duyệt. Giữ `check_gate` và shard/gold parity; mỗi lần chạy dùng thư mục run riêng. Phần cleaner/xuất cũ phía trên của notebook 01 chỉ để tham khảo và chẩn đoán, không thay cho lượt full v26 đã duyệt.

Cần đủ báo cáo `extraction_summary`, `missing`, `db_status_mismatch`, `domain_quality`, `duplicate_text_groups`, `offset_errors`. Đủ ID bao gồm placeholder không đồng nghĩa mọi ID đã có nội dung. `textproc/` và đồng bộ VPS/HF là các bước riêng; notebook 01 không xác nhận chúng đã hoàn tất. Docs đẩy lên dataset **private** đích của nhóm trước 24/10, phục vụ bài nộp 25/10; `AIGuruTinix/ViBioMIR` ở cell tải corpus là nguồn BTC.

### 5. VOV — sau các việc ưu tiên

Chạy cell Playwright probe một URL (`RUN_VOV_BROWSER=True`, `VOV_MAX_URLS=1`). Chỉ sau khi probe thành công mới đặt `VOV_MAX_URLS=None`. Ít nhất 2 giây/trang; gặp chặn dừng ngay và xem `vov_browser_probe.csv`.

### Chạy local

CLI `scripts/run_local.py` dùng cell notebook theo ID; `R2AI_DATA_ROOT` và `R2AI_STATUS_DB` chọn dữ liệu/DB hiện có. Các lệnh: `preflight`, `pubmed`, `crawl`, `vov`, `merge-home`, `extract`, `status`, `stop`. Xem `docs/local_run.md`.
